In [1]:
#!/usr/bin/env python3
"""Local Linux smoke runs using the repository's gcp_setup_stellar_private.sh.

Edit the settings below, then run: python3 run_local_protocols.py
Requires the repository's build dependencies. No GCP, commit or push operations.
All replicas share one host, so these are functional tests, not distributed benchmarks.
"""

import csv
import fcntl
import hashlib
import json
import os
from pathlib import Path
import re
import shlex
import signal
import socket
import subprocess
import tempfile
import time
from datetime import datetime
from urllib.request import build_opener, ProxyHandler


# ---------------- Settings: edit these ----------------
REPO = Path.home() / "stellar-core"
RESULTS_ROOT = Path.home() / "stellar-local-results"
NUM_NODES = 4                 # Start with 4; use 8 to match your GCP script.
PROTOCOLS = ["shabdiz", "iths", "pbft", "tetrabft", "simple_it", "forget_it", "scp"]
# Add "simple_it_bracha" to exercise Simple-IT's non-optimistic variant.
BUILD_JOBS = 8
REGENERATE_BUILD = False       # Needed on first run after the source-file split.
CONFIGURE_ARGS = None         # None reuses ./config.status --config, if present.
# For a fresh checkout, set your normal options, e.g. CONFIGURE_ARGS = [].
RESTORE_ORIGINAL_BUILD = False  # Restore header AND rebuild original mode afterward.
CXX = os.environ.get("CXX", "g++")

WORKLOAD = "WORKLOAD_W"       # Write-only exercises consensus for every request.
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 60
CLIENT_DRAIN_TIMEOUT_SEC = 15
STARTUP_TIMEOUT_SEC = 120
SEND_INTERVAL_US = 0
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]
SCP_OBSERVATION_SEC = 60       # Native diagnostic; no shab_client for SCP.
# ------------------------------------------------------

MODE_FLAGS = {
    "iths": "ITHS_MODE", "pbft": "PBFT_MODE", "scp": "ENABLE_SCP_TRACKING",
    "tetrabft": "TETRABFT_MODE", "simple_it": "SIMPLE_IT_MODE",
    "forget_it": "FORGET_IT_MODE", "simple_it_bracha": "SIMPLE_IT_MODE",
}
COMMIT_TAGS = {
    "shabdiz": ("[FAST COMMITTED]",),
    # Accept both the legacy and native blog adapter's commit labels.
    "iths": ("[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"),
    "pbft": ("[PBFT COMMITTED-LOCAL]",), "tetrabft": ("[TETRA COMMITTED]",),
    "simple_it": ("[SIMPLE-IT COMMITTED]",), "simple_it_bracha": ("[SIMPLE-IT COMMITTED]",),
    "forget_it": ("[FORGET-IT COMMITTED]",),
}
HTTP = build_opener(ProxyHandler({}))  # Local endpoints must bypass HTTP proxies.


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2) + "\n")


def replace_once(text, pattern, replacement):
    result, count = re.subn(pattern, lambda _: replacement, text, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Expected one match for {pattern!r}, found {count}; check repository changes")
    return result


def set_constant(text, name, value):
    prefix = re.search(rf"^inline constexpr [^\n=]+\b{re.escape(name)}\s*=", text,
                       re.MULTILINE)
    if prefix is None:
        raise RuntimeError(f"Missing experiment setting: {name}")
    return replace_once(text, rf"^inline constexpr [^\n=]+\b{re.escape(name)}\s*=\s*[^;]+;",
                        prefix.group(0) + " " + str(value) + ";")


def select_protocol(original, protocol):
    text = original
    for flag in set(MODE_FLAGS.values()):
        text = set_constant(text, flag, "true" if MODE_FLAGS.get(protocol) == flag else "false")
    for name, value in {
        "SIMPLE_IT_OPTIMISTIC": "false" if protocol == "simple_it_bracha" else "true",
        "NonPrune_MODE": "false", "currentWorkload": WORKLOAD,
        "SERVER_BATCH_SIZE": SERVER_BATCH_SIZE, "CLIENT_PORT": CLIENT_PORT,
    }.items():
        text = set_constant(text, name, value)
    return text


def stop_processes(processes):
    # Only process groups started by THIS script; never pkill other experiments.
    for proc in processes:
        if proc.poll() is None:
            try:
                os.killpg(proc.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    deadline = time.monotonic() + 5
    for proc in processes:
        try:
            proc.wait(timeout=max(0.01, deadline - time.monotonic()))
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()


def command(args, cwd, log):
    with log.open("a") as out:
        out.write("\n$ " + shlex.join(map(str, args)) + "\n")
        out.flush()
        proc = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = proc.wait()
            if code:
                raise RuntimeError(f"Command failed ({code}); see {log}")
        finally:
            stop_processes([proc])


def launch(args, cwd, log):
    with log.open("w") as out:
        return subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)


def ensure_alive(processes):
    for i, proc in enumerate(processes, 1):
        if proc.poll() is not None:
            raise RuntimeError(f"Replica {i} exited with code {proc.returncode}; inspect its logs")


def check_ports(ports):
    if len(set(ports)) != len(ports):
        raise RuntimeError("Peer, HTTP, and client ports overlap")
    for port in ports:
        with socket.socket() as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("0.0.0.0", port))
            except OSError as exc:
                raise RuntimeError(f"Port {port} is occupied; stop the previous local run") from exc


def local_setup_text(source, root):
    # Reuse your generator, including its seeds, validator lists and port arithmetic.
    text = replace_once(source, r"^STELLAR_CORE=.*$", "STELLAR_CORE=" + shlex.quote(str(REPO / "src/stellar-core")))
    text = replace_once(text, r"^BASE_DIR=.*$", "BASE_DIR=" + shlex.quote(str(root / "nodes")))
    text = replace_once(text, r"^mapfile -t NODE_IPS <.*$",
                        "mapfile -t NODE_IPS < " + shlex.quote(str(root / "localhost_ips.txt")))
    return text.replace('$STELLAR_CORE gen-seed', '"$STELLAR_CORE" gen-seed')


def prepare_nodes(root, setup_source):
    root.mkdir(parents=True)
    (root / "localhost_ips.txt").write_text("127.0.0.1\n" * NUM_NODES)
    setup = root / "setup-local.sh"
    setup.write_text(local_setup_text(setup_source, root))
    command(["bash", setup, "start"], root, root / "setup.log")
    configs, ports = [], []
    for i in range(1, NUM_NODES + 1):
        cfg = root / "nodes" / f"node{i}" / "stellar-core.cfg"
        text = cfg.read_text()
        # All added settings must precede the generator's TOML table headers.
        additions = {
            "SEND_CUSTOM_MESSAGE": "true" if i == 1 else "false",
            "ALLOW_LOCALHOST_FOR_TESTING": "true", "PUBLIC_HTTP_PORT": "false",
            "TARGET_PEER_CONNECTIONS": str(NUM_NODES - 1),
            "MAX_ADDITIONAL_PEER_CONNECTIONS": str(8 * NUM_NODES),
        }
        for key, value in additions.items():
            if re.search(rf"^{key}\s*=", text, re.MULTILINE):
                text = replace_once(text, rf"^{key}\s*=.*$", f"{key}={value}")
            else:
                text = f"{key}={value}\n" + text
        cfg.write_text(text)
        peer = int(re.search(r"^PEER_PORT=(\d+)", text, re.MULTILINE)[1])
        http = int(re.search(r"^HTTP_PORT=(\d+)", text, re.MULTILINE)[1])
        ports.extend([peer, http])
        configs.append((cfg, http))
    check_ports(ports + [CLIENT_PORT])
    # The generator's init branch continues on new-db failure. Check each command here.
    for cfg, _ in configs:
        command([REPO / "src/stellar-core", "new-db", "--conf", cfg], cfg.parent,
                cfg.parent / "init.log")
    return configs


def wait_for_network(replicas, configs, custom):
    deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
    while time.monotonic() < deadline:
        ensure_alive(replicas)
        logs = [(cfg.parent / "stellar-core.log").read_text(errors="replace")
                if (cfg.parent / "stellar-core.log").exists() else "" for cfg, _ in configs]
        if all("Shabdiz started" in log for log in logs):
            if not custom or f"Client listener started on port {CLIENT_PORT}" in logs[0]:
                return
        time.sleep(1)
    raise TimeoutError("Full peer connectivity/client readiness timed out; inspect node logs")


def info_snapshot(configs):
    results = []
    for _, port in configs:
        try:
            with HTTP.open(f"http://127.0.0.1:{port}/info", timeout=3) as response:
                results.append(json.load(response))
        except Exception as exc:
            results.append({"error": str(exc)})
    return results


def count_commits(path, protocol):
    text = path.read_text(errors="replace")
    return sum(text.count(tag) for tag in COMMIT_TAGS[protocol])


def client_metrics(paths):
    count, latency_sum = 0, 0.0
    for path in paths:
        for line in path.read_text(errors="replace").splitlines():
            if "[CLIENT_SEC]" not in line:
                continue
            fields = dict(re.findall(r"(\w+)=([^\s]+)", line))
            if 1 <= int(fields["sec"]) <= CLIENT_DURATION_SEC:
                n = int(fields["requests"])
                count += n
                if n:
                    latency_sum += n * float(fields["avg_latency_ms"])
    return {"acked_in_window": count, "throughput_ops_sec": count / CLIENT_DURATION_SEC,
            "mean_latency_ms": latency_sum / count if count else None}


def run_trial(protocol, root, setup_source, load):
    replicas, clients = [], []
    result = {"protocol": protocol, **(load or {}), "status": "failed"}
    try:
        configs = prepare_nodes(root, setup_source)
        save_json(root / "run_config.json", {
            **result, "num_nodes": NUM_NODES, "workload": WORKLOAD,
            "batch_size": SERVER_BATCH_SIZE, "client_duration_sec": CLIENT_DURATION_SEC,
            "client_port": CLIENT_PORT, "send_interval_us": SEND_INTERVAL_US,
            "aggregate_max_in_flight": (load["active_clients"] * load["client_threads"] *
                                        load["max_in_flight"]) if load else None,
        })
        for cfg, _ in configs:
            replicas.append(launch([REPO / "src/stellar-core", "run", "--conf", cfg],
                                   cfg.parent, cfg.parent / "console.log"))
        wait_for_network(replicas, configs, protocol != "scp")
        if protocol == "scp":
            save_json(root / "info-before.json", info_snapshot(configs))
            end = time.monotonic() + SCP_OBSERVATION_SEC
            while time.monotonic() < end:
                ensure_alive(replicas)
                time.sleep(1)
            save_json(root / "info-after.json", info_snapshot(configs))
            result.update(status="diagnostic_only", note=(
                "Native SCP workload creates accounts locally. Ledger advancement alone "
                "does not establish successful workload application; inspect node logs. "
                "Do not compare this run with custom-client throughput."))
        else:
            paths = []
            for i in range(load["active_clients"]):
                path = root / f"client-{i + 1}.log"
                paths.append(path)
                clients.append(launch([
                    root.parent / "shab_client", "127.0.0.1", CLIENT_PORT,
                    load["max_in_flight"], 100000000, SERVER_BATCH_SIZE,
                    CLIENT_DURATION_SEC, SEND_INTERVAL_US, load["client_threads"],
                ], root, path))
            end = time.monotonic() + CLIENT_DURATION_SEC + CLIENT_DRAIN_TIMEOUT_SEC
            while any(p.poll() is None for p in clients) and time.monotonic() < end:
                ensure_alive(replicas)
                time.sleep(0.5)
            ensure_alive(replicas)
            timed_out = any(p.poll() is None for p in clients)
            codes = [p.poll() for p in clients]
            stop_processes(clients)
            metrics = client_metrics(paths)
            per_client = [client_metrics([path])["acked_in_window"] for path in paths]
            commits = [count_commits(cfg.parent / "stellar-core.log", protocol)
                       for cfg, _ in configs]
            result.update(metrics, client_exit_codes=codes, per_client_acked=per_client,
                          per_replica_commit_log_counts=commits)
            if (any(code not in (None, 0) for code in codes) or not all(per_client) or
                    (WORKLOAD != "WORKLOAD_R100" and not all(commits))):
                raise RuntimeError("A client failed, a client received no ACKs, or a replica logged no commits")
            result["status"] = "drain_timeout" if timed_out else "ok"
            if timed_out:
                result["note"] = ("Client did not finish draining within the grace period. "
                                  "A final partial batch can cause this; inspect logs for stalls.")
    except Exception as exc:
        result.update(status="failed", error=str(exc))
    finally:
        stop_processes(clients)
        stop_processes(replicas)
        root.mkdir(parents=True, exist_ok=True)
        save_json(root / "result.json", result)
    return result


def write_summary(session, results):
    save_json(session / "summary.json", results)
    fields = ["protocol", "active_clients", "client_threads", "max_in_flight", "status",
              "acked_in_window", "throughput_ops_sec", "mean_latency_ms", "error", "note"]
    with (session / "summary.csv").open("w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)


def main():
    global REPO, RESULTS_ROOT
    REPO, RESULTS_ROOT = REPO.expanduser().resolve(), RESULTS_ROOT.expanduser().resolve()
    header = REPO / "src/overlay/experiments/ExperimentConfig.h"
    source = REPO / "gcp_setup_stellar_private.sh"
    if not header.is_file() or not source.is_file():
        raise RuntimeError("Set REPO to your updated stellar-core checkout")
    if NUM_NODES < 4 or not PROTOCOLS or not LOAD_POINTS:
        raise ValueError("Use at least four replicas and nonempty protocol/load lists")
    if CLIENT_DURATION_SEC <= 0 or SERVER_BATCH_SIZE <= 0 or BUILD_JOBS <= 0:
        raise ValueError("Duration, batch size and build jobs must be positive")
    if not re.fullmatch(r"WORKLOAD_(A|B|F|W|R0|R10|R20|R25|R40|R50|R60|R75|R80|R100)", WORKLOAD):
        raise ValueError("Unknown WORKLOAD")
    for protocol in PROTOCOLS:
        if protocol != "shabdiz" and protocol not in MODE_FLAGS:
            raise ValueError(f"Unknown protocol: {protocol}")
    for load in LOAD_POINTS:
        if min(load.values()) <= 0:
            raise ValueError("Client counts, threads and in-flight limits must be positive")
        if load["active_clients"] * load["client_threads"] * load["max_in_flight"] < SERVER_BATCH_SIZE:
            raise ValueError("Aggregate in-flight limit must cover at least one server batch")
    # Separate invocations must not race while changing this checkout's build flags.
    lockname = "stellar-local-" + hashlib.sha256(str(REPO).encode()).hexdigest()[:16] + ".lock"
    with (Path(tempfile.gettempdir()) / lockname).open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        session = RESULTS_ROOT / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        session.mkdir(parents=True)
        original = header.read_bytes()
        setup_source = source.read_text()
        (session / "ExperimentConfig.original.h").write_bytes(original)
        (session / "setup.original.sh").write_text(setup_source)
        command(["git", "rev-parse", "HEAD"], REPO, session / "source.log")
        command(["git", "status", "--short"], REPO, session / "source.log")
        command(["git", "diff", "HEAD"], REPO, session / "source.diff")
        print(f"Results: {session}", flush=True)
        results, changed, interrupted = [], False, False
        try:
            if REGENERATE_BUILD:
                options = CONFIGURE_ARGS
                if options is None:
                    if not (REPO / "config.status").exists():
                        raise RuntimeError("Fresh checkout: set CONFIGURE_ARGS to your configure options")
                    options = shlex.split(subprocess.check_output(
                        ["./config.status", "--config"], cwd=REPO, text=True))
                command(["./autogen.sh"], REPO, session / "bootstrap.log")
                command(["./configure", *options], REPO, session / "bootstrap.log")
            for protocol in PROTOCOLS:
                print(f"\nBuilding {protocol} ...", flush=True)
                folder = session / protocol
                folder.mkdir()
                selected = select_protocol(original.decode(), protocol)
                changed = True
                header.write_text(selected)
                (folder / "ExperimentConfig.h").write_text(selected)
                command(["make", f"-j{BUILD_JOBS}"], REPO, folder / "build.log")
                if protocol != "scp":
                    command([*shlex.split(CXX), "-O2", "-std=c++17", "-pthread",
                             "-I" + str(REPO / "src"), REPO / "shab_client.cpp",
                             "-o", folder / "shab_client"], REPO, folder / "build.log")
                for index, load in enumerate([None] if protocol == "scp" else LOAD_POINTS):
                    label = "native" if load is None else (
                        f"load{index + 1}_c{load['active_clients']}_t{load['client_threads']}_w{load['max_in_flight']}")
                    print(f"Running {protocol}/{label} ...", flush=True)
                    result = run_trial(protocol, folder / label, setup_source, load)
                    results.append(result)
                    write_summary(session, results)
                    print(json.dumps(result), flush=True)
        except KeyboardInterrupt:
            interrupted = True
            print("Interrupted; stopping this script's processes and restoring settings.", flush=True)
            raise
        finally:
            if changed:
                header.write_bytes(original)
                if RESTORE_ORIGINAL_BUILD and not interrupted:
                    print("Restoring original build ...", flush=True)
                    command(["make", f"-j{BUILD_JOBS}"], REPO, session / "restore-build.log")
                else:
                    print("Header restored. Run make before using src/stellar-core manually; "
                          "its binary may still use the last tested protocol.", flush=True)
        print(f"\nFinished. Summary: {session / 'summary.csv'}", flush=True)
        return 1 if any(r["status"] == "failed" for r in results) else 0


def interrupted_by_signal(*_):
    raise KeyboardInterrupt


if __name__ == "__main__":
    signal.signal(signal.SIGTERM, interrupted_by_signal)
    raise SystemExit(main())


Results: /home/tejas/stellar-local-results/20261105-183009-349449

Building shabdiz ...
Running shabdiz/load1_c1_t1_w150 ...
{"protocol": "shabdiz", "active_clients": 1, "client_threads": 1, "max_in_flight": 150, "status": "drain_timeout", "acked_in_window": 140300, "throughput_ops_sec": 2338.3333333333335, "mean_latency_ms": 64.02366856735567, "client_exit_codes": [null], "per_client_acked": [140300], "per_replica_commit_log_counts": [1404, 1404, 1404, 1404], "note": "Client did not finish draining within the grace period. A final partial batch can cause this; inspect logs for stalls."}
Running shabdiz/load2_c1_t2_w150 ...
{"protocol": "shabdiz", "active_clients": 1, "client_threads": 2, "max_in_flight": 150, "status": "drain_timeout", "acked_in_window": 2990600, "throughput_ops_sec": 49843.333333333336, "mean_latency_ms": 5.988167683408013, "client_exit_codes": [null], "per_client_acked": [2990600], "per_replica_commit_log_counts": [29909, 29909, 29909, 29909], "note": "Client did no

SystemExit: 0

/home/tejas/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# GCP Client Load: Throughput vs Latency (All Protocols)

In [ ]:
"""GCP throughput versus ACK latency; run this cell after editing the settings.
Builds and runs only when this cell is executed. No automatic git commit/push.
Reuses the image's configured build with incremental make; no autogen/configure.
Pins one source commit and preserves raw logs.
"""
import concurrent.futures
import fcntl
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import re
import shlex
import socket
import subprocess
import tarfile
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_ROOT = Path.home() / "work/experiments/gcp-client-load"
RESUME_SESSION = "latest"              # Rerun to resume; None starts fresh, or set a session folder path.
# Resume keeps the original source commit/settings; use None after changing the experiment.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-load"
INSTANCE_PREFIX = "tsm-sc-load"         # Dedicated fleet for this experiment.
NUM_NODES = 4                          # Forget-IT requires 3f+1: 4, 7, 10, ...
REPLICA_ZONES = ["us-central1-c"] * NUM_NODES
CLIENT_ZONES = ["us-central1-c", "us-central1-c"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Room for an isolated build and run data.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = False     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # Match e2-standard-2; build on replica 1.
INSTALL_ZLIB_DEV = True                # New Shabdiz archive needs zlib1g-dev.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOLS = ["shabdiz", "iths", "pbft", "tetrabft", "simple_it", "forget_it"]
# simple_it = Opt-RBC, non-speculative. Optionally add "simple_it_bracha".
# SCP has a different native workload/ACK interface and is not comparable here.
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 360
CLIENT_DRAIN_TIMEOUT_SEC = 45
MEASUREMENT_START_SEC = 30              # ACK completion window [30, 360).
MEASUREMENT_END_SEC = CLIENT_DURATION_SEC
STARTUP_TIMEOUT_SEC = 180
CLIENT_START_LEAD_SEC = 45              # Common wall-clock start across client VMs.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
ITHS_BLOG_DELTA_MS = 1000               # Explicit: 2 Delta wait EACH sequential batch.
FORGET_IT_DELTA_MS = 1000               # Recovery timing; preserves current default.
SHABDIZ_FAST_PATH_ONLY = False
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
    {"active_clients": 1, "client_threads": 4, "max_in_flight": 150},
    {"active_clients": 1, "client_threads": 8, "max_in_flight": 150},
    {"active_clients": 2, "client_threads": 8, "max_in_flight": 150},
    {"active_clients": 2, "client_threads": 16, "max_in_flight": 150},
]
# By default VMs remain running for reuse. Deletion applies only to this run\'s fleet.
# ------------------------------------------------------

MODE_FLAGS = {"iths": "ITHS_MODE", "pbft": "PBFT_MODE", "scp": "ENABLE_SCP_TRACKING",
              "tetrabft": "TETRABFT_MODE", "simple_it": "SIMPLE_IT_MODE",
              "simple_it_bracha": "SIMPLE_IT_MODE", "forget_it": "FORGET_IT_MODE"}
COMMIT_TAGS = {"shabdiz": ["[FAST COMMITTED]"],
               "iths": ["[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"],
               "pbft": ["[PBFT COMMITTED-LOCAL]"], "tetrabft": ["[TETRA COMMITTED]"],
               "simple_it": ["[SIMPLE-IT COMMITTED]"],
               "simple_it_bracha": ["[SIMPLE-IT COMMITTED]"],
               "forget_it": ["[FORGET-IT COMMITTED]"]}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def select_protocol(original, protocol):
    values = {flag: MODE_FLAGS.get(protocol) == flag for flag in set(MODE_FLAGS.values())}
    values.update(SIMPLE_IT_OPTIMISTIC=protocol != "simple_it_bracha", NonPrune_MODE=False,
                  currentWorkload=WORKLOAD, SERVER_BATCH_SIZE=SERVER_BATCH_SIZE,
                  CLIENT_PORT=CLIENT_PORT, ITHS_BLOG_DELTA_MS=ITHS_BLOG_DELTA_MS,
                  FORGET_IT_DELTA_MS=FORGET_IT_DELTA_MS,
                  SHABDIZ_FAST_PATH_ONLY=SHABDIZ_FAST_PATH_ONLY,
                  # No forced collection during this fault-free experiment.
                  FORCE_COLLECT_AFTER_SEC=2147483647)
    for name, value in values.items():
        value = str(value).lower() if isinstance(value, bool) else str(value)
        pattern = rf"^(inline constexpr [^\n=]+\b{re.escape(name)}\s*=)\s*[^;]+;"
        original, count = re.subn(pattern, lambda m: m[1] + " " + value + ";",
                                  original, flags=re.MULTILINE)
        if count != 1:
            raise ValueError(f"Missing/duplicate build constant: {name}")
    return original


# A bounded remote supervisor. It only stops the child process group it created.
# On controller interruption, each process still has its own maximum lifetime.
WORKER_SOURCE = r'''
import json, os, signal, subprocess, sys, time
from pathlib import Path

def write(path, value):
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value))
    tmp.replace(path)

def state(root):
    done = root / 'done.json'
    if done.exists():
        return json.loads(done.read_text())
    status = {'done': False, 'ready': False, 'listener': False}
    logfile = root / 'stellar-core.log'
    if logfile.exists():
        with logfile.open('rb') as f:
            f.seek(max(0, logfile.stat().st_size - 131072))
            text = f.read().decode(errors='replace')
        status['ready'] = 'Shabdiz started' in text
        status['listener'] = 'Client listener started' in text
    return status

mode, root_arg = sys.argv[1:3]
root = Path(root_arg)
if mode == 'launch':
    spec = json.loads(sys.argv[3])
    root.mkdir(parents=True, exist_ok=True)
    if (root / 'spec.json').exists():
        raise RuntimeError('Refusing to overwrite an existing process directory')
    write(root / 'spec.json', spec)
    with (root / 'supervisor.log').open('w') as log:
        subprocess.Popen([sys.executable, __file__, 'supervise', str(root)],
                         stdin=subprocess.DEVNULL, stdout=log, stderr=log,
                         start_new_session=True, close_fds=True)
elif mode == 'supervise':
    spec = json.loads((root / 'spec.json').read_text())
    proc = None
    outcome = {'done': True, 'timed_out': False, 'stopped': False, 'returncode': None}
    try:
        while time.time() < spec.get('start_at', 0):
            if (root / 'stop').exists():
                raise RuntimeError('Stopped before scheduled start')
            time.sleep(0.05)
        env = dict(os.environ)
        env['LD_LIBRARY_PATH'] = spec['lib'] + ':' + env.get('LD_LIBRARY_PATH', '')
        with (root / spec['log']).open('w') as out:
            outcome['started_at'] = time.time()
            proc = subprocess.Popen(spec['args'], cwd=root, env=env,
                                    stdin=subprocess.DEVNULL, stdout=out, stderr=out,
                                    start_new_session=True)
            end = time.monotonic() + spec['timeout']
            while proc.poll() is None:
                outcome['stopped'] = (root / 'stop').exists()
                outcome['timed_out'] = time.monotonic() >= end
                if outcome['stopped'] or outcome['timed_out']:
                    try:
                        os.killpg(proc.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        pass
                    try:
                        proc.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        os.killpg(proc.pid, signal.SIGKILL)
                    break
                time.sleep(0.25)
            outcome['returncode'] = proc.wait()
    except BaseException as exc:
        outcome['error'] = str(exc)
    finally:
        if proc is not None and proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
        write(root / 'done.json', outcome)
elif mode == 'state':
    print(json.dumps(state(root)))
elif mode == 'stop':
    if (root / 'spec.json').exists():
        (root / 'stop').touch()
        until = time.monotonic() + 12
        while not (root / 'done.json').exists() and time.monotonic() < until:
            time.sleep(0.2)
        if not (root / 'done.json').exists():
            raise RuntimeError('Supervisor did not acknowledge stop; inspect ' + str(root))
else:
    raise ValueError(mode)
'''


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    vms = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] == "TERMINATED":
                    cloud("instances", "start", name, "--zone", zone)
                elif old["status"] != "RUNNING":
                    raise RuntimeError(f"{name} is {old['status']}")
            elif CREATE_MISSING_VMS:
                cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                      "--image", image["selfLink"], "--network", NETWORK,
                      "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                      "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
            else:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
            if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
                raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
            if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
                cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
            disk_url = vm["disks"][0]["source"]
            disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                    "--zone", zone, "--format=json"))
            if disk.get("sourceImage") != image["selfLink"]:
                raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
            vms.append({"name": name, "zone": zone, "role": role, "index": index,
                        "ip": vm["networkInterfaces"][0]["networkIP"],
                        "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]})
            save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def worker(vm, action, root, spec=None):
    args = ["python3", REMOTE_SESSION + "/worker.py", action, root]
    if spec is not None:
        args.append(json.dumps(spec))
    return ssh(vm, shlex.join(args), timeout=60)


def build_protocol(builder, protocol, header, folder):
    folder.mkdir(parents=True, exist_ok=True)
    selected = select_protocol(header, protocol)
    (folder / "ExperimentConfig.h").write_text(selected)
    scp(builder, folder / "ExperimentConfig.h", REMOTE_BUILD + "/src/overlay/experiments/ExperimentConfig.h")
    art = REMOTE_SESSION + "/artifacts/" + protocol
    # Include non-system shared libraries from the image build, if any.
    package_code = f'''
import hashlib, json, os, pathlib, shutil, subprocess
root = pathlib.Path({art!r}); root.mkdir(parents=True, exist_ok=True)
(root/'lib').mkdir(exist_ok=True)
subprocess.run(['strip', '-o', str(root/'stellar-core'), 'src/stellar-core'], check=True)
shutil.copy2('shab_client', root/'shab_client')
shutil.copy2('src/overlay/experiments/ExperimentConfig.h', root/'ExperimentConfig.h')
queue = [root/'stellar-core', root/'shab_client']; seen = set()
while queue:
    binary = queue.pop()
    output = subprocess.check_output(['ldd', str(binary)], text=True)
    if 'not found' in output: raise RuntimeError(output)
    for line in output.splitlines():
        fields = line.split()
        if '=>' not in fields: continue
        lib = pathlib.Path(fields[fields.index('=>')+1])
        if not lib.is_absolute() or str(lib).startswith(('/lib/', '/lib64/', '/usr/lib/')): continue
        if str(lib) in seen: continue
        seen.add(str(lib)); shutil.copy2(lib, root/'lib'/lib.name); queue.append(lib)
(root/'binaries.json').write_text(json.dumps({{p.name:hashlib.sha256(p.read_bytes()).hexdigest()
                                           for p in [root/'stellar-core', root/'shab_client']}}, indent=2))
'''
    # Preserve generated Makefiles/config.status, including in recursive makes.
    # Checkout timestamps must not trigger Automake regeneration or configure.
    script = (f"cd {shlex.quote(REMOTE_BUILD)}\n"
              f"make -j{BUILD_JOBS} -o Makefile -o config.status "
              "MAKE='make -o Makefile -o config.status'\n"
              f"g++ -O2 -std=c++17 -pthread -Isrc shab_client.cpp -o shab_client\n"
              + "python3 -c " + shlex.quote(package_code) + "\n"
              + shlex.join(["tar", "-czf", art + ".tgz", "-C", art, "."]))
    # A disconnected SSH client may leave make running; never start a competing build.
    ssh(builder, shlex.join(['flock', '-n', REMOTE_ROOT + '/build.lock', 'bash', '-lc',
                            'set -euo pipefail\n' + script]), log=folder / "build.log", timeout=14400)
    scp(builder, art + ".tgz", folder / "binaries.tgz", download=True)
    return art


def deploy(vm, protocol, folder, art):
    digest = file_digest(folder / "binaries.tgz")
    check = f'''
import json, pathlib, hashlib
root = pathlib.Path({art!r})
try:
    manifest = json.loads((root/'binaries.json').read_text())
    ready = ((root/'.deployed').read_text().strip() == {digest!r}
             and all(hashlib.sha256((root/name).read_bytes()).hexdigest() == manifest[name]
                     for name in ('stellar-core', 'shab_client')))
except (OSError, ValueError, KeyError):
    ready = False
print('ready' if ready else 'missing')
'''
    if python_remote(vm, check).strip() == 'ready':
        print('Reusing deployment:', vm['name'], protocol, flush=True)
        return
    ssh(vm, shlex.join(["mkdir", "-p", art]))
    scp(vm, folder / "binaries.tgz", art + ".tgz")
    ssh(vm, shlex.join(["tar", "-xzf", art + ".tgz", "-C", art]) + "\n" +
        "LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + " ldd " + shlex.quote(art + "/stellar-core") +
        " | tee " + shlex.quote(art + "/ldd.txt") + "\n! grep -q 'not found' " + shlex.quote(art + "/ldd.txt") +
        "\nprintf '%s\\n' " + shlex.quote(digest) + " > " + shlex.quote(art + "/.deployed"))


def make_configs(builder, art, trial, node_ips, folder):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\npython3 -c " + shlex.quote(code),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def run_trial(vms, builder, art, protocol, load, repetition, load_index, session, common):
    global FLEET_CLEANUP_FAILED
    label = trial_label(protocol, load, repetition, load_index)
    folder = session / label
    archive_failed_trial(vms, folder, label, session)
    folder.mkdir(parents=True)
    # Each retry gets a new remote directory; old supervisors/logs remain identifiable.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    try:
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join([art + "/stellar-core", "new-db", "--conf", root + "/stellar-core.cfg"]),
                log=folder / f"node-{vm['index']}-init.log")
            worker(vm, "launch", root, {"args": [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"],
                   "lib": art + "/lib", "log": "console.log",
                   "timeout": STARTUP_TIMEOUT_SEC + CLIENT_START_LEAD_SEC + CLIENT_DURATION_SEC + CLIENT_DRAIN_TIMEOUT_SEC + 300})
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        while True:
            status = parallel(lambda v: json.loads(worker(v, "state", trial + f"/nodes/node{v['index']}")), nodes)
            if any(s['done'] for s in status):
                raise RuntimeError("Replica exited during startup")
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                raise TimeoutError("Full peer connectivity/client listener timed out")
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC
        config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            worker(vm, "launch", trial + f"/client-{vm['index']}", {
                "args": [art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                         "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                         str(SEND_INTERVAL_US), str(load['client_threads'])],
                "lib": art + "/lib", "log": "client.log", "start_at": start_at,
                "timeout": CLIENT_DURATION_SEC + CLIENT_DRAIN_TIMEOUT_SEC})
        parallel(start_client, clients)
        if time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        deadline = time.monotonic() + CLIENT_START_LEAD_SEC + CLIENT_DURATION_SEC + CLIENT_DRAIN_TIMEOUT_SEC + 90
        while True:
            states = parallel(lambda v: json.loads(worker(v, "state", trial + f"/client-{v['index']}")), clients)
            replica_states = parallel(lambda v: json.loads(worker(v, "state", trial + f"/nodes/node{v['index']}")), nodes)
            if any(s['done'] for s in replica_states):
                raise RuntimeError("Replica exited during workload")
            if all(s['done'] for s in states):
                break
            if time.monotonic() >= deadline:
                raise TimeoutError("Remote client supervisors did not finish")
            time.sleep(10)
        result['client_outcomes'] = states
        starts = [s['started_at'] for s in states]
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or s['stopped'] or (s['returncode'] != 0 and not s['timed_out']) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'drain_timeout' if any(s['timed_out'] for s in states) else 'ok'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        for vm, root in owned:
            try:
                worker(vm, 'stop', root)
            except Exception as exc:
                FLEET_CLEANUP_FAILED = True
                cleanup_errors.append(f"{vm['name']}: {exc}")
        def collect(pair):
            vm, root = pair
            local = folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            local.mkdir(exist_ok=True)
            # Only logs/config/status: do not transfer buckets or archive.sqlite.
            for filename in (['stellar-core.log', 'console.log', 'stellar-core.cfg', 'done.json']
                             if vm['role']=='node' else ['client.log', 'done.json']):
                scp(vm, root + '/' + filename, local / filename, download=True)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        if result['status'] in ('ok', 'drain_timeout'):
            counts = [sum((folder / f"node-{v['index']}" / 'stellar-core.log').read_text(errors='replace').count(t)
                          for t in COMMIT_TAGS[protocol]) for v in nodes]
            result['replica_commits'] = counts
            if not all(counts):
                result.update(status='failed', error='A replica logged no commits')
        save_json(folder / "result.json", result)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError('Cleanup/download failed; stopped sweep. Inspect ' + str(folder))
    return result


def file_digest(path):
    with Path(path).open('rb') as stream:
        return stream_digest(stream)


def stream_digest(stream):
    digest = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
    return digest.hexdigest()


def cached_build(folder, selected_header):
    # Validate the archive itself: an interrupted download must not count as a build.
    try:
        with tarfile.open(folder / 'binaries.tgz', 'r:gz') as archive:
            members = {m.name.removeprefix('./'): m for m in archive.getmembers() if m.isfile()}
            def read(name):
                return archive.extractfile(members[name])
            if read('ExperimentConfig.h').read().decode() != selected_header:
                return False
            manifest = json.load(read('binaries.json'))
            return all(stream_digest(read(name)) == manifest[name]
                       for name in ('stellar-core', 'shab_client'))
    except (OSError, EOFError, tarfile.TarError, ValueError, KeyError, UnicodeError):
        return False


def reuse_build(builder, protocol, folder, selected_header):
    if cached_build(folder, selected_header):
        return True
    # Recover a finished remote build if the previous invocation lost its download.
    remote = REMOTE_SESSION + '/artifacts/' + protocol + '.tgz'
    if ssh(builder, 'if [ -f ' + shlex.quote(remote) + ' ]; then echo present; fi').strip() == 'present':
        folder.mkdir(parents=True, exist_ok=True)
        scp(builder, remote, folder / 'binaries.tgz', download=True)
        return cached_build(folder, selected_header)
    return False


def trial_label(protocol, load, repetition, index):
    return (f"{protocol}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def completed_result(session, trial, common):
    protocol, repetition, index, load = trial
    label = trial_label(protocol, load, repetition, index)
    folder = session / label
    try:
        result = json.loads((folder / 'result.json').read_text())
        config = json.loads((folder / 'run_config.json').read_text())
        expected = {**common, **load, 'protocol': protocol, 'repetition': repetition,
                    'load_index': index}
        # New resume metadata is not present in legacy run_config.json files.
        if any(config.get(k) != v for k, v in expected.items()
               if k not in RESUME_METADATA_KEYS):
            return None
        if result.get('status') not in ('ok', 'drain_timeout') or result.get('path') != label:
            return None
        for role, count, filenames in [
            ('node', NUM_NODES, ('stellar-core.log', 'console.log', 'stellar-core.cfg', 'done.json')),
            ('client', load['active_clients'], ('client.log', 'done.json')),
        ]:
            if any(not (folder / f'{role}-{i}' / name).is_file()
                   for i in range(1, count + 1) for name in filenames):
                return None
        return result
    except (OSError, ValueError):
        return None


def archive_failed_trial(vms, folder, label, session):
    if not folder.exists():
        return
    config_path = folder / 'run_config.json'
    if config_path.exists():
        prior = json.loads(config_path.read_text())
        root = prior['remote_trial']
        if not root.startswith(REMOTE_SESSION + '/runs/'):
            raise RuntimeError('Prior trial belongs to another remote session: ' + root)
        # Confirm old processes stopped before reusing ports. A failed stop keeps the lock.
        global FLEET_CLEANUP_FAILED
        try:
            for vm in vms:
                if vm['role'] == 'node':
                    worker(vm, 'stop', root + f"/nodes/node{vm['index']}")
                elif vm['index'] <= prior['active_clients']:
                    worker(vm, 'stop', root + f"/client-{vm['index']}")
        except BaseException:
            FLEET_CLEANUP_FAILED = True
            raise
    destination = session / 'failed-attempts' / uuid.uuid4().hex / label
    destination.parent.mkdir(parents=True)
    folder.rename(destination)
    print('Previous attempt saved:', destination, flush=True)


def session_to_resume():
    if RESUME_SESSION is None:
        return None
    if str(RESUME_SESSION) == 'latest':
        candidates = sorted(RESULTS_ROOT.expanduser().resolve().glob('*/session.json'))
        return candidates[-1].parent if candidates else None
    session = Path(RESUME_SESSION).expanduser().resolve()
    if not (session / 'session.json').is_file():
        raise FileNotFoundError('No session.json in ' + str(session))
    return session


def acquire_fleet_lock(vm):
    owner = {'session': REMOTE_SESSION, 'controller': CONTROLLER_LOCK_ID}
    code = f'''
import json, pathlib
lock = pathlib.Path({REMOTE_ROOT + '/active.lock'!r})
lock.parent.mkdir(parents=True, exist_ok=True)
owner = {owner!r}
try:
    lock.mkdir()
except FileExistsError:
    try:
        previous = json.loads((lock/'owner.json').read_text())
    except (OSError, ValueError):
        previous = None
    if previous != owner:
        raise RuntimeError('Fleet lock belongs to another controller/session, or predates resume support: '
                           + str(lock) + '. Confirm the earlier controller has stopped before clearing it.')
# The local flock proves no other invocation on this controller owns this lock.
(lock/'owner.json').write_text(json.dumps(owner))
'''
    python_remote(vm, code)


def release_fleet_lock(vm):
    code = f'''
import json, pathlib
lock = pathlib.Path({REMOTE_ROOT + '/active.lock'!r})
expected = {{'session': {REMOTE_SESSION!r}, 'controller': {CONTROLLER_LOCK_ID!r}}}
if json.loads((lock/'owner.json').read_text()) != expected:
    raise RuntimeError('Fleet lock owner changed; refusing to remove it')
(lock/'owner.json').unlink()
lock.rmdir()
'''
    python_remote(vm, code)


def prepare_build(builder, sha, session):
    # Mark readiness only after all preparation succeeds; retries reuse the same copy.
    marker = REMOTE_BUILD + '/.gcp-load-prepared'
    script = f'''
if [ -f {shlex.quote(marker)} ] && [ "$(cat {shlex.quote(marker)})" = {shlex.quote(sha)} ] && \\
   [ -f {shlex.quote(REMOTE_BUILD + '/Makefile')} ] && [ -f {shlex.quote(REMOTE_BUILD + '/config.status')} ]; then
    echo 'Reusing prepared build tree'
    exit 0
fi
test -d {shlex.quote(IMAGE_REPO + '/.git')}
mkdir -p {shlex.quote(REMOTE_BUILD)}
cp -a --reflink=auto {shlex.quote(IMAGE_REPO + '/.')} {shlex.quote(REMOTE_BUILD + '/')}
cd {shlex.quote(REMOTE_BUILD)}
export GIT_TERMINAL_PROMPT=0
git fetch origin {shlex.quote(sha)}
git reset --hard {shlex.quote(sha)}
git submodule sync --recursive
git submodule update --init --recursive
'''
    if INSTALL_ZLIB_DEV:
        script += 'sudo -n apt-get update\nsudo -n apt-get install -y zlib1g-dev\n'
    script += (
        'if [ ! -f Makefile ] || [ ! -f config.status ]; then\n'
        '  echo "Image checkout must contain a configured build; this experiment only runs make." >&2\n'
        '  exit 1\nfi\n'
        + "printf '%s\\n' " + shlex.quote(sha) + ' > ' + shlex.quote(marker) + '\n'
    )
    ssh(builder, shlex.join(['flock', '-n', REMOTE_ROOT + '/build.lock', 'bash', '-lc',
                            'set -euo pipefail\n' + script]),
        log=session / 'prepare-build.log', timeout=14400)


def finish_session(session):
    print('Finished:', session, '\nRun the GCP client-load plotting cell in PostProcess.ipynb.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        locked, deleted = [], set()
        try:
            for vm in remaining:
                acquire_fleet_lock(vm)
                locked.append(vm)
            def delete(vm):
                cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900)
                deleted.add(vm['name'])
            parallel(delete, remaining)
        finally:
            for vm in locked:
                if vm['name'] not in deleted:
                    release_fleet_lock(vm)
        print('All instances in this run have been deleted.')
    else:
        print('VMs are kept for reuse:', ', '.join(v['name'] for v in saved['vms']))


RESUME_METADATA_KEYS = {'project', 'image_project', 'remote_root', 'image_repo', 'replica_zones',
                        'client_zones', 'client_port', 'client_drain_timeout_sec'}


def main_gcp_load():
    global REMOTE_SESSION, REMOTE_BUILD, FLEET_CLEANUP_FAILED
    FLEET_CLEANUP_FAILED = False
    repo = REPO.expanduser().resolve()
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if NUM_NODES < 4 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and at least four replicas")
    if 'forget_it' in PROTOCOLS and (NUM_NODES - 1) % 3:
        raise ValueError("Forget-IT requires n=3f+1 (4, 7, 10, ...); 8 is unsupported")
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - COMMIT_TAGS.keys():
        raise ValueError("Choose distinct custom-client protocols from COMMIT_TAGS; SCP is diagnostic only")
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError("Invalid measurement window")
    if min(REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError("Counts, batch size, and Delta must be positive")
    if not re.fullmatch(r'WORKLOAD_(W|R0)', WORKLOAD):
        raise ValueError("This comparison uses write-only transactions; choose WORKLOAD_W or WORKLOAD_R0")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    session = session_to_resume()
    previous = json.loads((session / 'session.json').read_text()) if session else None
    # Resume the original source even after a notebook-only pull advances local HEAD.
    source_ref = SOURCE_COMMIT or (previous['source_commit'] if previous else 'HEAD')
    sha = command(['git', '-C', repo, 'rev-parse', source_ref + '^{commit}']).strip()
    header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
    # Fail before provisioning if source settings have changed.
    for protocol in PROTOCOLS:
        select_protocol(header, protocol)
    if session is None:
        session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
        session = RESULTS_ROOT.expanduser().resolve() / session_id
    else:
        session_id = previous['session']
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = REMOTE_SESSION + '/repo'
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'workload': WORKLOAD, 'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': CLIENT_DRAIN_TIMEOUT_SEC}
    manifest = {**common, 'protocols': PROTOCOLS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN}
    if previous is not None:
        changed = [key for key in manifest if key in previous and manifest[key] != previous[key]
                   and key != 'delete_all_after_final_run']
        if (session / 'fleet.json').exists():
            old_vms = json.loads((session / 'fleet.json').read_text())['vms']
            for role, zones in [('node', REPLICA_ZONES), ('client', CLIENT_ZONES)]:
                if any(v['index'] > len(zones) or zones[v['index']-1] != v['zone']
                       for v in old_vms if v['role'] == role):
                    changed.append(role + '_zones')
        if changed:
            raise ValueError('Resume settings differ: ' + ', '.join(changed) +
                             '. Restore those settings or set RESUME_SESSION = None for a new session.')
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', session, '\nSource:', sha, flush=True)
    trials = [(protocol, repetition, index, load) for protocol in PROTOCOLS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = [completed_result(session, trial, common) for trial in trials]
    results = [result for result in results if result is not None]
    completed = {result['path'] for result in results}
    pending = [trial for trial in trials if trial_label(trial[0], trial[3], trial[1], trial[2]) not in completed]
    save_json(session / 'results.json', results)
    print(f'Reusing {len(completed)} completed trials; {len(pending)} pending.', flush=True)
    if not pending:
        finish_session(session)
        return session
    print(f"{len(pending)} remaining trials; "
          f"{len(pending)*(CLIENT_DURATION_SEC+CLIENT_DRAIN_TIMEOUT_SEC)/3600:.1f} "
          'hours of workload/drain, plus builds and setup.', flush=True)
    print(f'IT-HS Delta={ITHS_BLOG_DELTA_MS} ms: its per-batch proposal wait is {2*ITHS_BLOG_DELTA_MS} ms.')
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    (session / 'worker.py').write_text(WORKER_SOURCE)
    locked = []
    try:
        # Atomic per-fleet lock prevents concurrent notebook invocations using this fleet.
        for vm in vms:
            acquire_fleet_lock(vm)
            locked.append(vm)
            ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]))
            scp(vm, session / 'worker.py', REMOTE_SESSION + '/worker.py')
        for vm in vms:
            sync = ssh(vm, 'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before running IT-HS/scheduled clients")
        # Stop all incomplete attempts before changing binaries or starting another protocol.
        for protocol, repetition, index, load in pending:
            label = trial_label(protocol, load, repetition, index)
            archive_failed_trial(vms, session / label, label, session)
        builder = next(v for v in vms if v['role']=='node')
        prepare_build(builder, sha, session)
        for protocol in PROTOCOLS:
            protocol_trials = [trial for trial in pending if trial[0] == protocol]
            if not protocol_trials:
                print('Skipping completed protocol:', protocol, flush=True)
                continue
            print('Building/deploying', protocol, flush=True)
            build_folder = session / 'builds' / protocol
            art = REMOTE_SESSION + '/artifacts/' + protocol
            if reuse_build(builder, protocol, build_folder, select_protocol(header, protocol)):
                print('Reusing cached build:', protocol, flush=True)
            else:
                art = build_protocol(builder, protocol, header, build_folder)
            parallel(lambda v: deploy(v, protocol, build_folder, art), vms)
            for _, repetition, index, load in protocol_trials:
                result = run_trial(vms, builder, art, protocol, load, repetition, index, session, common)
                results.append(result)
                save_json(session / 'results.json', results)
    finally:
        for vm in locked:
            if FLEET_CLEANUP_FAILED:
                print('Stop not confirmed; retaining fleet lock on', vm['name'])
                continue
            try:
                release_fleet_lock(vm)
            except Exception as exc:
                print('Could not release lock on', vm['name'], exc)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs are kept for retry. Rerun this cell to resume:', session)
    else:
        finish_session(session)
    return session


def run_gcp_load():
    global CONTROLLER_LOCK_ID
    root = RESULTS_ROOT.expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    lock_path = root / '.gcp-client-load-controller.lock'
    # Held for the entire cell; released by Python on exceptions or kernel exit.
    with lock_path.open('a') as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('Another GCP client-load cell is still running for this results folder')
        CONTROLLER_LOCK_ID = socket.gethostname() + ':' + str(lock_path)
        try:
            return main_gcp_load()
        finally:
            fcntl.flock(lock, fcntl.LOCK_UN)


GCP_LOAD_SESSION = run_gcp_load()


# Main num nodes experiment

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]
# # NUM_NODES_LIST = [32]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = True

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = True

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 210
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j4; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     # if num_nodes >= 2:
#     #     target_file = "../stellar-private/node2/stellar-core.cfg"
#     #     line_to_add = "MEMORY_PROF=true"

#     #     subprocess.call(
#     #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#     #         shell=True
#     #     )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"PBFT_vs_num_nodes_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # After this run, delete only the machines that the next smaller run will
#     # not need. Example:
#     #   after 48-node run, keep 000..032 and delete 033..048
#     #   after 32-node run, keep 000..016 and delete 017..032
#     #   after 16-node run, keep 000..008 and delete 009..016
#     #   after 8-node run,  keep 000..004 and delete 005..008
#     # Final run optionally deletes everything.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(48, len(instances_to_delete))
#             )
#             print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")


# Failure Experiment

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [16]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 180
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         # results = run_parallel(
#         #     compile_stellar,
#         #     range(num_nodes + n_clients),
#         #     max_workers=min(48, num_nodes + n_clients)
#         # )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     # if num_nodes >= 2:
#     #     target_file = "../stellar-private/node2/stellar-core.cfg"
#     #     line_to_add = "MEMORY_PROF=true"

#     #     subprocess.call(
#     #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#     #         shell=True
#     #     )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         # time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         time.sleep(100)

#         results = run_parallel(
#             kill_stellar_private,
#             [4],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)

        


#         results = run_parallel(
#             kill_stellar_private,
#             [5],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)


#         results = run_parallel(
#             kill_stellar_private,
#             [6],
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [7],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [8],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(80)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"Failure_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # After this run, delete only the machines that the next smaller run will
#     # not need. Example:
#     #   after 48-node run, keep 000..032 and delete 033..048
#     #   after 32-node run, keep 000..016 and delete 017..032
#     #   after 16-node run, keep 000..008 and delete 009..016
#     #   after 8-node run,  keep 000..004 and delete 005..008
#     # Final run optionally deletes everything.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(32, len(instances_to_delete))
#             )
#             print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")


# Collection vs Collection Rounds Experiment

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path


# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


# ============================================================
# Experiment config
# ============================================================

# latencies: 50, 90, 150, 210
default_region = ["us-west1-b"]
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
regions = ["us-west1-b", "us-west1-b", "us-west1-b", "us-west1-b"]

zone_no = 0

# System size for this failure / collection-window experiment.
NUM_NODES_LIST = [16]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Collection-round sweep.
COLLECT_ATTEMPT_POINTS = [300]

# Source file to patch before each subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Start clean only once, before the first run.
DELETE_BEFORE_FIRST_RUN = True

# Delete VMs after each MAX_COLLECT_ATTEMPTS subrun.
DELETE_UNUSED_AFTER_EACH_RUN = True

# Delete remaining VMs after final run.
# This is mostly redundant if DELETE_UNUSED_AFTER_EACH_RUN=True.
DELETE_ALL_AFTER_FINAL_RUN = True

# IMPORTANT:
# Code changes across collection-round settings, so we must compile every subrun.
COMPILE_ONLY_FIRST_RUN = False

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Main experiment loop
# ============================================================

for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
    for collect_idx, max_collect_attempts in enumerate(COLLECT_ATTEMPT_POINTS):

        project = "research-488322"
        zone = "us-west1-b"
        machine_type = "e2-standard-2"
        image_family = "tsm-sc-family"
        subnet = "default"
        gcp_username = "tejas"

        print("\n" + "#" * 100)
        print(
            f"Starting experiment for num_nodes={num_nodes}, "
            f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
        )
        print("#" * 100)

        def get_zone_for_instance(i):
            # Use MAX_NUM_NODES so zone assignment stays stable.
            if i < int(MAX_NUM_NODES / 2):
                return default_region[0]
            else:
                return regions[zone_no]

        def parse_tsm_index(name):
            return int(name.rsplit("-", 1)[1])

        # -----------------------------------------------------------------
        # Delete existing tsm-sc-* instances only before the first subrun,
        # if enabled.
        # -----------------------------------------------------------------
        if DELETE_BEFORE_FIRST_RUN and run_idx == 0 and collect_idx == 0:
            print("\n🧹 Cleaning before first run...")
            delete_all_tsm_instances(project)

        # -----------------------------------------------------------------
        # Create only missing replica/client machines for this subrun.
        # -----------------------------------------------------------------
        existing_instances = fetch_existing_instances(project)
        existing_names = {name for name, _ in existing_instances}

        commands = []
        newly_created_indices = []

        # Create replica nodes only if they do not already exist.
        for i in range(num_nodes):
            instance_name = f"tsm-sc-{i:03}"

            if instance_name in existing_names:
                print(f"✔ Reusing existing replica {instance_name}")
                continue

            inst_zone = get_zone_for_instance(i)

            cmd = f'''
            gcloud compute instances create {instance_name} \
                --project={project} \
                --zone={inst_zone} \
                --machine-type={machine_type} \
                --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
                --can-ip-forward \
                --maintenance-policy=MIGRATE \
                --provisioning-model=STANDARD \
                --service-account=254510644191-compute@developer.gserviceaccount.com \
                --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
                --tags=http-server,https-server \
                --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
                --no-shielded-secure-boot \
                --shielded-vtpm \
                --shielded-integrity-monitoring \
                --labels=goog-ec-src=vm_add-gcloud \
                --reservation-affinity=any
            '''
            commands.append(cmd.strip())
            newly_created_indices.append(i)

        # Create client machine only if it does not already exist.
        for i in range(n_clients):
            client_idx = num_nodes + i
            instance_name = f"tsm-sc-{client_idx:03}"

            if instance_name in existing_names:
                print(f"✔ Reusing existing client {instance_name}")
                continue

            inst_zone = get_zone_for_instance(client_idx)

            cmd = f'''
            gcloud compute instances create {instance_name} \
                --project={project} \
                --zone={inst_zone} \
                --machine-type={machine_type} \
                --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
                --can-ip-forward \
                --maintenance-policy=MIGRATE \
                --provisioning-model=STANDARD \
                --service-account=254510644191-compute@developer.gserviceaccount.com \
                --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
                --tags=http-server,https-server \
                --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
                --no-shielded-secure-boot \
                --shielded-vtpm \
                --shielded-integrity-monitoring \
                --labels=goog-ec-src=vm_add-gcloud \
                --reservation-affinity=any
            '''
            commands.append(cmd.strip())
            newly_created_indices.append(client_idx)

        if commands:
            run_parallel(
                run_command,
                commands,
                max_workers=min(48, len(commands)),
            )

            print("All missing instances launched.")

            # Give GCP/SSH a little time after VM creation.
            time.sleep(30)
        else:
            print("✔ All required instances already exist; no VM creation needed.")

        # -----------------------------------------------------------------
        # Get sorted node and client IPs.
        # tsm_ips.txt must include only replica IPs, not clients.
        # -----------------------------------------------------------------
        ip_cmd = f'''
        gcloud compute instances list \
            --project={project} \
            --filter="name~'^tsm-sc-'" \
            --sort-by=name \
            --format="value(name,zone,networkInterfaces[0].networkIP)"
        '''

        ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

        instance_records = []

        for line in ip_output.splitlines():
            if line.strip():
                name, inst_zone, ip = line.split()
                idx = int(name.rsplit("-", 1)[1])
                instance_records.append((idx, name, inst_zone, ip))

        instance_records.sort()

        node_records = [r for r in instance_records if r[0] < num_nodes]
        client_records = [
            r for r in instance_records
            if num_nodes <= r[0] < num_nodes + n_clients
        ]

        if len(node_records) != num_nodes:
            raise RuntimeError(
                f"Expected {num_nodes} replica nodes, but found "
                f"{len(node_records)}: {node_records}"
            )

        if len(client_records) != n_clients:
            raise RuntimeError(
                f"Expected {n_clients} client nodes, but found "
                f"{len(client_records)}: {client_records}"
            )

        iplist = [r[3] for r in node_records]

        with open("tsm_ips.txt", "w") as f:
            for ip in iplist:
                f.write(ip + "\n")

        print("🎯 Node IPs:", iplist)
        print("🎯 Client instances:", client_records)

        node1_ip = iplist[0]
        print(f"Clients will connect to leader/node1 at: {node1_ip}")

        # -----------------------------------------------------------------
        # Remote helper functions.
        # -----------------------------------------------------------------

        def kill_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        def git_pull_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        def compile_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        def clean_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        def copy_folder_to_instance(
            i,
            source_folder="/home/tejas/stellar-private",
            destination_path="/home/tejas/stellar-private",
        ):
            inst_zone = get_zone_for_instance(i)
            instance_name = f"tsm-sc-{i:03}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

            print(f"Executing command for {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Command for {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        def run_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            node_number = i + 1
            instance_name = f"tsm-sc-{i:03}"

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        def run_stellar_client(i, client_max_in_flight, client_threads):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing client command: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # -----------------------------------------------------------------
        # Generate stellar-private configs locally using only replica IPs.
        # -----------------------------------------------------------------
        stellar_private_path = Path("../stellar-private")
        if stellar_private_path.exists():
            shutil.rmtree(stellar_private_path)
        stellar_private_path.mkdir()

        subprocess.call(
            "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
            shell=True,
        )

        subprocess.call(
            "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
            "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
            shell=True,
        )

        # Enable custom message only on leader.
        line_to_add = "SEND_CUSTOM_MESSAGE=true"
        target_file = "../stellar-private/node1/stellar-core.cfg"

        subprocess.call(
            f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
            shell=True,
        )

        print("\n" + "*" * 100)
        print(f"Preparing subrun with MAX_COLLECT_ATTEMPTS={max_collect_attempts}")
        print("*" * 100)

        # Patch local source.
        patch_max_collect_attempts(max_collect_attempts)

        # Push patched source.
        push_collect_patch_to_git(max_collect_attempts)

        # Stop any old processes.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Pull patched code on active VMs.
        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Compile patched code on active VMs.
        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # -----------------------------------------------------------------
        # Throughput/latency loop.
        #
        # IMPORTANT:
        # This is now inside the MAX_COLLECT_ATTEMPTS loop.
        # So every collect-attempt value gets its own actual run.
        # -----------------------------------------------------------------
        for load in LOAD_POINTS:
            active_clients = load["active_clients"]
            client_threads = load["client_threads"]
            client_max_in_flight = load["max_in_flight"]

            if active_clients > n_clients:
                raise RuntimeError(
                    f"active_clients={active_clients} exceeds n_clients={n_clients}"
                )

            total_client_threads = active_clients * client_threads
            aggregate_max_in_flight = (
                active_clients * client_threads * client_max_in_flight
            )

            run_label = (
                f"clients_{active_clients}_threads_{client_threads}_"
                f"inflight_{client_max_in_flight}_"
                f"total_threads_{total_client_threads}_"
                f"total_inflight_{aggregate_max_in_flight}"
            )

            print("\n" + "=" * 80)
            print(
                f"🚀 Starting run: num_nodes={num_nodes}, "
                f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, {run_label}"
            )
            print("=" * 80)

            # Clean remote stellar-private.
            results = run_parallel(
                clean_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Copy fresh stellar-private config to all active machines.
            results = run_parallel(
                copy_folder_to_instance,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Kill old processes on all active machines.
            results = run_parallel(
                kill_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Start consensus replicas.
            results = run_parallel(
                run_stellar_private,
                range(num_nodes),
                max_workers=min(48, num_nodes),
            )

            print(results)
            print("All Stellar nodes should be starting in the background.")

            # Give nodes time to authenticate and start the client listener.
            time.sleep(40)

            # Start only required client VMs for this load point.
            active_client_indices = [
                num_nodes + j for j in range(active_clients)
            ]

            results = run_parallel(
                lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
                active_client_indices,
                max_workers=active_clients,
            )

            print(results)
            print(
                f"Started {active_clients} client VM(s), "
                f"each with {client_threads} client threads. "
                f"Total client threads = {total_client_threads}. "
                f"Aggregate max in-flight = {aggregate_max_in_flight}."
            )

            # Let the system run.
            time.sleep(CLIENT_WAIT_AFTER_START_SEC)

            # Stop all nodes and clients.
            results = run_parallel(
                kill_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # -------------------------------------------------------------
            # Save logs.
            # -------------------------------------------------------------
            remote_base_folder = "/home/tejas/stellar-private"

            local_base_destination = (
                "/home/tejas/work/experiments/shabdiz/"
                + f"Collection_{max_collect_attempts}_"
                + f"nodes_{num_nodes}_{run_label}"
            )

            Path(local_base_destination).mkdir(parents=True, exist_ok=True)

            # Save run metadata.
            with open(Path(local_base_destination) / "run_config.txt", "w") as f:
                f.write(f"num_nodes={num_nodes}\n")
                f.write(f"max_collect_attempts={max_collect_attempts}\n")
                f.write(f"force_collect_after_sec=100\n")
                f.write(f"active_clients={active_clients}\n")
                f.write(f"client_threads_per_vm={client_threads}\n")
                f.write(f"total_client_threads={total_client_threads}\n")
                f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
                f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
                f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
                f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
                f.write(f"leader_ip={node1_ip}\n")
                f.write(f"machine_type={machine_type}\n")
                f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

            def copy_folder_from_instance(i):
                inst_zone = get_zone_for_instance(i)

                instance_name = f"tsm-sc-{i:03}"

                node_number = i + 1
                node_folder = f"node{node_number}"

                remote_source_path = posixpath.join(remote_base_folder, node_folder)

                local_destination_path = Path(local_base_destination) / instance_name
                local_destination_path.mkdir(parents=True, exist_ok=True)

                remote_source = f"{instance_name}:{remote_source_path}"

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

                print(
                    f"Executing command to copy {node_folder} "
                    f"from {instance_name}: {command}"
                )

                output = subprocess.call(command, shell=True)

                print(f"Copy from {instance_name} finished with exit code: {output}")

                return (instance_name, output)

            # Copy all replica logs.
            # FIX: was range(3), which copied only nodes 0, 1, 2.
            node_copy_results = run_parallel(
                copy_folder_from_instance,
                range(3),
                max_workers=3,
            )

            def copy_client_log(i):
                inst_zone = get_zone_for_instance(i)

                instance_name = f"tsm-sc-{i:03}"
                client_id = i - num_nodes

                remote_source = (
                    f"{instance_name}:/home/tejas/stellar-private/"
                    f"stellar-client-{client_id}.log"
                )

                local_destination_path = Path(local_base_destination)
                local_destination_path.mkdir(parents=True, exist_ok=True)

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

                print(f"Copying client log from {instance_name}...")

                output = subprocess.call(command, shell=True)

                print(f"Copy finished with exit code: {output}")

                return (instance_name, output)

            client_copy_results = run_parallel(
                copy_client_log,
                active_client_indices,
                max_workers=active_clients,
            )

            print("\n--- Summary of Download Results ---")
            print("Node log copies:", node_copy_results)
            print("Client log copies:", client_copy_results)
            print(f"Saved run to: {local_base_destination}")

        # -----------------------------------------------------------------
        # Delete instances after this MAX_COLLECT_ATTEMPTS subrun.
        #
        # IMPORTANT:
        # This is inside the collect-attempt loop, so after Collection_50
        # finishes, VMs are deleted before Collection_300 starts.
        # -----------------------------------------------------------------
        if DELETE_UNUSED_AFTER_EACH_RUN:
            print(
                "\n🧹 Deleting VMs after completed subrun: "
                f"num_nodes={num_nodes}, MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
            )
            delete_all_tsm_instances(project)


# ============================================================
# Final cleanup
# ============================================================

if DELETE_ALL_AFTER_FINAL_RUN and not DELETE_UNUSED_AFTER_EACH_RUN:
    print("\n🧹 Final cleanup after all runs...")
    delete_all_tsm_instances(project)

# Collection vs num nodes

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path


# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def parse_tsm_index(name):
    return int(name.rsplit("-", 1)[1])


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


def prune_tsm_instances_for_next_run(project, next_num_nodes, n_clients):
    """
    Keep only instances needed for the next system size.

    For next_num_nodes = N and n_clients = 1, we keep:
        replicas: tsm-sc-000 ... tsm-sc-(N-1)
        client:   tsm-sc-N

    Everything with index > N is deleted.
    """
    keep_indices = set(range(next_num_nodes + n_clients))
    instances = fetch_existing_instances(project)

    instances_to_delete = []

    for name, inst_zone in instances:
        idx = parse_tsm_index(name)
        if idx not in keep_indices:
            instances_to_delete.append((name, inst_zone))

    print(
        "\n➡ Pruning instances for next run: "
        f"next_num_nodes={next_num_nodes}, keeping indices "
        f"0 through {next_num_nodes + n_clients - 1}"
    )

    if instances_to_delete:
        print("Instances to delete:")
        for name, inst_zone in instances_to_delete:
            print(f"  - {name} ({inst_zone})")

        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances_to_delete,
            max_workers=min(32, len(instances_to_delete)),
        )

        print("\n🧹 Non-needed instances deleted.\n")
    else:
        print("\n✔ No non-needed instances to delete.\n")


# ============================================================
# Experiment config
# ============================================================

# latencies: 50, 90, 150, 210
default_region = ["us-west1-b"]
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
regions = ["us-west1-b", "us-west1-b", "us-west1-b", "us-west1-b"]

zone_no = 0

project = "research-488322"
zone = "us-west1-b"
machine_type = "e2-standard-2"
image_family = "tsm-sc-family"
subnet = "default"
gcp_username = "tejas"

# Vary system size.
# Descending order is important because we can create 32 once,
# compile once, and then delete unnecessary higher-index nodes.
NUM_NODES_LIST = [4]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Fixed collection attempts.
MAX_COLLECT_ATTEMPTS = 100

# Source file to patch before the first subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Start clean only once, before the first run.
DELETE_BEFORE_FIRST_RUN = False

# After each subrun, delete nodes not needed for the next smaller run.
PRUNE_UNUSED_AFTER_EACH_RUN = True

# Delete remaining VMs after the final run.
DELETE_ALL_AFTER_FINAL_RUN = True

# Code does not change across system sizes now, so compile only first subrun.
COMPILE_ONLY_FIRST_RUN = True

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Main experiment loop
# ============================================================

for run_idx, num_nodes in enumerate(NUM_NODES_LIST):

    print("\n" + "#" * 100)
    print(
        f"Starting experiment for num_nodes={num_nodes}, "
        f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}"
    )
    print("#" * 100)

    def get_zone_for_instance(i):
        # Use MAX_NUM_NODES so zone assignment stays stable.
        if i < int(MAX_NUM_NODES / 2):
            return default_region[0]
        else:
            return regions[zone_no]

    # -----------------------------------------------------------------
    # Delete existing tsm-sc-* instances only before the first subrun.
    # -----------------------------------------------------------------
    if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
        print("\n🧹 Cleaning before first run...")
        delete_all_tsm_instances(project)

    # -----------------------------------------------------------------
    # Create only missing replica/client machines for this system size.
    # -----------------------------------------------------------------
    existing_instances = fetch_existing_instances(project)
    existing_names = {name for name, _ in existing_instances}

    commands = []
    newly_created_indices = []

    # Create replica nodes only if they do not already exist.
    for i in range(num_nodes):
        instance_name = f"tsm-sc-{i:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing replica {instance_name}")
            continue

        inst_zone = get_zone_for_instance(i)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(i)

    # Create client machine only if it does not already exist.
    for i in range(n_clients):
        client_idx = num_nodes + i
        instance_name = f"tsm-sc-{client_idx:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing client {instance_name}")
            continue

        inst_zone = get_zone_for_instance(client_idx)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(client_idx)

    if commands:
        run_parallel(
            run_command,
            commands,
            max_workers=min(48, len(commands)),
        )

        print("All missing instances launched.")

        # Give GCP/SSH a little time after VM creation.
        time.sleep(30)
    else:
        print("✔ All required instances already exist; no VM creation needed.")

    # -----------------------------------------------------------------
    # Get sorted node and client IPs.
    # tsm_ips.txt must include only replica IPs, not clients.
    # -----------------------------------------------------------------
    ip_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --sort-by=name \
        --format="value(name,zone,networkInterfaces[0].networkIP)"
    '''

    ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

    instance_records = []

    for line in ip_output.splitlines():
        if line.strip():
            name, inst_zone, ip = line.split()
            idx = int(name.rsplit("-", 1)[1])
            instance_records.append((idx, name, inst_zone, ip))

    instance_records.sort()

    node_records = [r for r in instance_records if r[0] < num_nodes]
    client_records = [
        r for r in instance_records
        if num_nodes <= r[0] < num_nodes + n_clients
    ]

    if len(node_records) != num_nodes:
        raise RuntimeError(
            f"Expected {num_nodes} replica nodes, but found "
            f"{len(node_records)}: {node_records}"
        )

    if len(client_records) != n_clients:
        raise RuntimeError(
            f"Expected {n_clients} client nodes, but found "
            f"{len(client_records)}: {client_records}"
        )

    iplist = [r[3] for r in node_records]

    with open("tsm_ips.txt", "w") as f:
        for ip in iplist:
            f.write(ip + "\n")

    print("🎯 Node IPs:", iplist)
    print("🎯 Client instances:", client_records)

    node1_ip = iplist[0]
    print(f"Clients will connect to leader/node1 at: {node1_ip}")

    # -----------------------------------------------------------------
    # Remote helper functions.
    # -----------------------------------------------------------------

    def kill_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    def git_pull_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    def compile_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    def clean_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    def copy_folder_to_instance(
        i,
        source_folder="/home/tejas/stellar-private",
        destination_path="/home/tejas/stellar-private",
    ):
        inst_zone = get_zone_for_instance(i)
        instance_name = f"tsm-sc-{i:03}"

        command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

        print(f"Executing command for {instance_name}: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Command for {instance_name} finished with exit code: {output}")

        return (instance_name, output)

    def run_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        node_number = i + 1
        instance_name = f"tsm-sc-{i:03}"

        remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Return code for {instance_name}: {output}")
        return output

    def run_stellar_client(i, client_max_in_flight, client_threads):
        inst_zone = get_zone_for_instance(i)

        instance_name = f"tsm-sc-{i:03}"
        client_id = i - num_nodes

        remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing client command: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Return code for {instance_name}: {output}")
        return output

    # -----------------------------------------------------------------
    # Generate stellar-private configs locally using only replica IPs.
    # -----------------------------------------------------------------
    stellar_private_path = Path("../stellar-private")
    if stellar_private_path.exists():
        shutil.rmtree(stellar_private_path)
    stellar_private_path.mkdir()

    subprocess.call(
        "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
        shell=True,
    )

    subprocess.call(
        "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
        "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
        shell=True,
    )

    # Enable custom message only on leader.
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg"

    subprocess.call(
        f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
        shell=True,
    )

    print("\n" + "*" * 100)
    print(
        f"Preparing subrun with num_nodes={num_nodes}, "
        f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}"
    )
    print("*" * 100)

    # -----------------------------------------------------------------
    # Patch/push/compile only on the first subrun.
    #
    # Because MAX_COLLECT_ATTEMPTS is fixed at 50 and system sizes are
    # run in descending order, all later nodes are reused from the first
    # 32-node deployment and already have compiled code.
    # -----------------------------------------------------------------
    if run_idx == 0 or not COMPILE_ONLY_FIRST_RUN:
        # Patch local source.
        patch_max_collect_attempts(MAX_COLLECT_ATTEMPTS)

        # Push patched source.
        push_collect_patch_to_git(MAX_COLLECT_ATTEMPTS)

        # Stop any old processes.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Pull patched code on active VMs.
        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Compile patched code on active VMs.
        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)
    else:
        print(
            "\n✔ Skipping git pull and compile because "
            "COMPILE_ONLY_FIRST_RUN=True and this is not the first subrun."
        )

        # Still kill any stale processes on reused machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

    # -----------------------------------------------------------------
    # Throughput/latency loop.
    # -----------------------------------------------------------------
    for load in LOAD_POINTS:
        active_clients = load["active_clients"]
        client_threads = load["client_threads"]
        client_max_in_flight = load["max_in_flight"]

        if active_clients > n_clients:
            raise RuntimeError(
                f"active_clients={active_clients} exceeds n_clients={n_clients}"
            )

        total_client_threads = active_clients * client_threads
        aggregate_max_in_flight = (
            active_clients * client_threads * client_max_in_flight
        )

        run_label = (
            f"clients_{active_clients}_threads_{client_threads}_"
            f"inflight_{client_max_in_flight}_"
            f"total_threads_{total_client_threads}_"
            f"total_inflight_{aggregate_max_in_flight}"
        )

        print("\n" + "=" * 80)
        print(
            f"🚀 Starting run: num_nodes={num_nodes}, "
            f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}, {run_label}"
        )
        print("=" * 80)

        # Clean remote stellar-private.
        results = run_parallel(
            clean_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Copy fresh stellar-private config to all active machines.
        results = run_parallel(
            copy_folder_to_instance,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Kill old processes on all active machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Start consensus replicas.
        results = run_parallel(
            run_stellar_private,
            range(num_nodes),
            max_workers=min(48, num_nodes),
        )

        print(results)
        print("All Stellar nodes should be starting in the background.")

        # Give nodes time to authenticate and start the client listener.
        time.sleep(60)

        # Start only required client VMs for this load point.
        active_client_indices = [
            num_nodes + j for j in range(active_clients)
        ]

        results = run_parallel(
            lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
            active_client_indices,
            max_workers=active_clients,
        )

        print(results)
        print(
            f"Started {active_clients} client VM(s), "
            f"each with {client_threads} client threads. "
            f"Total client threads = {total_client_threads}. "
            f"Aggregate max in-flight = {aggregate_max_in_flight}."
        )

        # Let the system run.
        time.sleep(CLIENT_WAIT_AFTER_START_SEC)

        # Stop all nodes and clients.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # -------------------------------------------------------------
        # Save logs.
        # -------------------------------------------------------------
        remote_base_folder = "/home/tejas/stellar-private"

        local_base_destination = (
            "/home/tejas/work/experiments/shabdiz/"
            + f"CollectionVSNumNodes_{MAX_COLLECT_ATTEMPTS}_"
            + f"nodes_{num_nodes}_{run_label}"
        )

        Path(local_base_destination).mkdir(parents=True, exist_ok=True)

        # Save run metadata.
        with open(Path(local_base_destination) / "run_config.txt", "w") as f:
            f.write(f"num_nodes={num_nodes}\n")
            f.write(f"max_collect_attempts={MAX_COLLECT_ATTEMPTS}\n")
            f.write(f"force_collect_after_sec=100\n")
            f.write(f"active_clients={active_clients}\n")
            f.write(f"client_threads_per_vm={client_threads}\n")
            f.write(f"total_client_threads={total_client_threads}\n")
            f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
            f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
            f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
            f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
            f.write(f"leader_ip={node1_ip}\n")
            f.write(f"machine_type={machine_type}\n")
            f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

        def copy_folder_from_instance(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"

            node_number = i + 1
            node_folder = f"node{node_number}"

            remote_source_path = posixpath.join(remote_base_folder, node_folder)

            local_destination_path = Path(local_base_destination) / instance_name
            local_destination_path.mkdir(parents=True, exist_ok=True)

            remote_source = f"{instance_name}:{remote_source_path}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

            print(
                f"Executing command to copy {node_folder} "
                f"from {instance_name}: {command}"
            )

            output = subprocess.call(command, shell=True)

            print(f"Copy from {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        # Copy only first 3 replica logs.
        node_copy_results = run_parallel(
            copy_folder_from_instance,
            range(3),
            max_workers=3,
        )

        def copy_client_log(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_source = (
                f"{instance_name}:/home/tejas/stellar-private/"
                f"stellar-client-{client_id}.log"
            )

            local_destination_path = Path(local_base_destination)
            local_destination_path.mkdir(parents=True, exist_ok=True)

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

            print(f"Copying client log from {instance_name}...")

            output = subprocess.call(command, shell=True)

            print(f"Copy finished with exit code: {output}")

            return (instance_name, output)

        client_copy_results = run_parallel(
            copy_client_log,
            active_client_indices,
            max_workers=active_clients,
        )

        print("\n--- Summary of Download Results ---")
        print("Node log copies:", node_copy_results)
        print("Client log copies:", client_copy_results)
        print(f"Saved run to: {local_base_destination}")

    # -----------------------------------------------------------------
    # After each system-size subrun, delete nodes not needed for the next.
    # -----------------------------------------------------------------
    is_last_run = run_idx == len(NUM_NODES_LIST) - 1

    if PRUNE_UNUSED_AFTER_EACH_RUN and not is_last_run:
        next_num_nodes = NUM_NODES_LIST[run_idx + 1]
        prune_tsm_instances_for_next_run(
            project=project,
            next_num_nodes=next_num_nodes,
            n_clients=n_clients,
        )

    elif is_last_run and DELETE_ALL_AFTER_FINAL_RUN:
        print("\n🧹 Final run completed. Deleting all remaining VMs...")
        delete_all_tsm_instances(project)

# Collection vs Network Latency

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path

import os
# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


# ============================================================
# Experiment config
# ============================================================

project = "research-488322"
machine_type = "e2-standard-2"
image_family = "tsm-sc-family"
subnet = "default"
gcp_username = "tejas"

# Fixed system size.
NUM_NODES = 16

# One client VM.
# Client index will be tsm-sc-016.
n_clients = 1

# Fixed collection attempts.
COLLECT_ATTEMPT_POINTS = [5]

# First 8 replicas are always here.
BASE_REPLICA_ZONE = "us-west1-b"

# Client is always here.
CLIENT_ZONE = "us-west1-b"

# Replicas 8..15 sweep over these remote zones.
REMOTE_REPLICA_ZONES = [
    # "asia-northeast1-b",
    "europe-west3-c",
    # "asia-south1-c",
    "northamerica-northeast2-b",
    # "us-west1-b"
]

# Source file to patch before each subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Local stellar-private path.
LOCAL_STELLAR_PRIVATE_PATH = Path("../stellar-private").resolve()

# Delete before first run.
DELETE_BEFORE_FIRST_RUN = True

# Delete all VMs after every subrun.
DELETE_AFTER_EACH_SUBRUN = True

# Since VMs are deleted after every subrun, compile every subrun.
COMPILE_EVERY_SUBRUN = True

# Copy only these replica logs.
# Change to [0, 8, 15] if you want one local and two remote-node logs.
COPY_NODE_INDICES = [0, 1, 2]

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add .; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Zone assignment
# ============================================================

def get_zone_for_instance(i, remote_replica_zone):
    """
    Replica layout for NUM_NODES = 16:
        tsm-sc-000 ... tsm-sc-007 -> us-west1-b
        tsm-sc-008 ... tsm-sc-015 -> remote_replica_zone
        tsm-sc-016                -> us-west1-b client
    """
    if i < 8:
        return BASE_REPLICA_ZONE

    if i < NUM_NODES:
        return remote_replica_zone

    return CLIENT_ZONE


# ============================================================
# GCP instance creation
# ============================================================

def create_instance_command(instance_name, inst_zone):
    return f'''
    gcloud compute instances create {instance_name} \
        --project={project} \
        --zone={inst_zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=254510644191-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''.strip()


# ============================================================
# Main experiment loop
# ============================================================
os.system('git add .; git commit -m "update"; git push')
subrun_idx = 0

for remote_replica_zone in REMOTE_REPLICA_ZONES:
    for max_collect_attempts in COLLECT_ATTEMPT_POINTS:

        subrun_idx += 1

        print("\n" + "#" * 100)
        print(
            f"Starting network-latency subrun {subrun_idx}: "
            f"num_nodes={NUM_NODES}, "
            f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
            f"first_8_zone={BASE_REPLICA_ZONE}, "
            f"next_8_zone={remote_replica_zone}, "
            f"client_zone={CLIENT_ZONE}"
        )
        print("#" * 100)

        if DELETE_BEFORE_FIRST_RUN and subrun_idx == 1:
            print("\n🧹 Cleaning before first run...")
            delete_all_tsm_instances(project)

        try:
            # ------------------------------------------------------------
            # Create all replica/client machines for this subrun.
            # Since we delete after every subrun, these are usually fresh.
            # ------------------------------------------------------------
            existing_instances = fetch_existing_instances(project)
            existing_names = {name for name, _ in existing_instances}

            commands = []

            # Create replica nodes.
            for i in range(NUM_NODES):
                instance_name = f"tsm-sc-{i:03}"

                if instance_name in existing_names:
                    print(f"✔ Reusing existing replica {instance_name}")
                    continue

                inst_zone = get_zone_for_instance(i, remote_replica_zone)
                commands.append(create_instance_command(instance_name, inst_zone))

            # Create client node.
            for i in range(n_clients):
                client_idx = NUM_NODES + i
                instance_name = f"tsm-sc-{client_idx:03}"

                if instance_name in existing_names:
                    print(f"✔ Reusing existing client {instance_name}")
                    continue

                inst_zone = get_zone_for_instance(client_idx, remote_replica_zone)
                commands.append(create_instance_command(instance_name, inst_zone))

            if commands:
                run_parallel(
                    run_command,
                    commands,
                    max_workers=min(48, len(commands)),
                )

                print("All missing instances launched.")
                time.sleep(30)
            else:
                print("✔ All required instances already exist; no VM creation needed.")

            # ------------------------------------------------------------
            # Get sorted replica and client IPs.
            # tsm_ips.txt must include only replica IPs, not clients.
            # ------------------------------------------------------------
            ip_cmd = f'''
            gcloud compute instances list \
                --project={project} \
                --filter="name~'^tsm-sc-'" \
                --sort-by=name \
                --format="value(name,zone,networkInterfaces[0].networkIP)"
            '''

            ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

            instance_records = []

            for line in ip_output.splitlines():
                if line.strip():
                    name, inst_zone, ip = line.split()
                    idx = int(name.rsplit("-", 1)[1])
                    instance_records.append((idx, name, inst_zone, ip))

            instance_records.sort()

            node_records = [r for r in instance_records if r[0] < NUM_NODES]
            client_records = [
                r for r in instance_records
                if NUM_NODES <= r[0] < NUM_NODES + n_clients
            ]

            if len(node_records) != NUM_NODES:
                raise RuntimeError(
                    f"Expected {NUM_NODES} replica nodes, but found "
                    f"{len(node_records)}: {node_records}"
                )

            if len(client_records) != n_clients:
                raise RuntimeError(
                    f"Expected {n_clients} client nodes, but found "
                    f"{len(client_records)}: {client_records}"
                )

            iplist = [r[3] for r in node_records]

            with open("tsm_ips.txt", "w") as f:
                for ip in iplist:
                    f.write(ip + "\n")

            print("🎯 Node IPs:", iplist)
            print("🎯 Client instances:", client_records)

            node1_ip = iplist[0]
            print(f"Clients will connect to leader/node1 at: {node1_ip}")

            # ------------------------------------------------------------
            # Remote helper functions.
            # ------------------------------------------------------------

            def kill_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")
                output = subprocess.call(command, shell=True)
                print(f"Return code for tsm-sc-{i:03}: {output}")
                return output

            def git_pull_stellar(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

                print(command)
                output = subprocess.call(command, shell=True)
                print(output)
                return output

            def compile_stellar(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

                print(command)
                output = subprocess.call(command, shell=True)
                print(output)
                return output

            def clean_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")
                output = subprocess.call(command, shell=True)
                print(f"Return code for tsm-sc-{i:03}: {output}")
                return output

            def copy_folder_to_instance(
                i,
                source_folder=None,
                destination_path="/home/tejas/stellar-private",
            ):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)
                instance_name = f"tsm-sc-{i:03}"

                if source_folder is None:
                    source_folder = str(LOCAL_STELLAR_PRIVATE_PATH)

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

                print(f"Executing command for {instance_name}: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Command for {instance_name} finished with exit code: {output}")

                return (instance_name, output)

            def run_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                node_number = i + 1
                instance_name = f"tsm-sc-{i:03}"

                remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Return code for {instance_name}: {output}")
                return output

            def run_stellar_client(i, client_max_in_flight, client_threads):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                instance_name = f"tsm-sc-{i:03}"
                client_id = i - NUM_NODES

                remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing client command: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Return code for {instance_name}: {output}")
                return output

            # ------------------------------------------------------------
            # Generate stellar-private configs locally using only replica IPs.
            # This must run every subrun because IPs/zones change.
            # ------------------------------------------------------------
            if LOCAL_STELLAR_PRIVATE_PATH.exists():
                shutil.rmtree(LOCAL_STELLAR_PRIVATE_PATH)
            LOCAL_STELLAR_PRIVATE_PATH.mkdir(parents=True, exist_ok=True)

            subprocess.call(
                f"cp gcp_setup_stellar_private.sh "
                f"{LOCAL_STELLAR_PRIVATE_PATH}/gcp_setup_stellar_private.sh",
                shell=True,
            )

            subprocess.call(
                f"cd {LOCAL_STELLAR_PRIVATE_PATH}; "
                f"chmod +x gcp_setup_stellar_private.sh; "
                f"./gcp_setup_stellar_private.sh start; "
                f"./gcp_setup_stellar_private.sh",
                shell=True,
            )

            # Enable custom message only on leader.
            line_to_add = "SEND_CUSTOM_MESSAGE=true"
            target_file = LOCAL_STELLAR_PRIVATE_PATH / "node1" / "stellar-core.cfg"

            subprocess.call(
                f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
                shell=True,
            )

            print("\n" + "*" * 100)
            print(
                f"Preparing subrun with "
                f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
                f"remote_replica_zone={remote_replica_zone}"
            )
            print("*" * 100)

            # ------------------------------------------------------------
            # Patch, push, pull, and compile every subrun.
            # ------------------------------------------------------------
            patch_max_collect_attempts(max_collect_attempts)
            push_collect_patch_to_git(max_collect_attempts)

            results = run_parallel(
                kill_stellar_private,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            results = run_parallel(
                git_pull_stellar,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            results = run_parallel(
                compile_stellar,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            # ------------------------------------------------------------
            # Throughput/latency loop.
            # ------------------------------------------------------------
            for load in LOAD_POINTS:
                active_clients = load["active_clients"]
                client_threads = load["client_threads"]
                client_max_in_flight = load["max_in_flight"]

                if active_clients > n_clients:
                    raise RuntimeError(
                        f"active_clients={active_clients} exceeds n_clients={n_clients}"
                    )

                total_client_threads = active_clients * client_threads
                aggregate_max_in_flight = (
                    active_clients * client_threads * client_max_in_flight
                )

                run_label = (
                    f"clients_{active_clients}_threads_{client_threads}_"
                    f"inflight_{client_max_in_flight}_"
                    f"total_threads_{total_client_threads}_"
                    f"total_inflight_{aggregate_max_in_flight}"
                )

                print("\n" + "=" * 80)
                print(
                    f"🚀 Starting run: "
                    f"num_nodes={NUM_NODES}, "
                    f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
                    f"remote_replica_zone={remote_replica_zone}, "
                    f"{run_label}"
                )
                print("=" * 80)

                # Clean remote stellar-private.
                results = run_parallel(
                    clean_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Copy fresh stellar-private config to all active machines.
                results = run_parallel(
                    copy_folder_to_instance,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Kill old processes on all active machines.
                results = run_parallel(
                    kill_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Start consensus replicas.
                results = run_parallel(
                    run_stellar_private,
                    range(NUM_NODES),
                    max_workers=min(48, NUM_NODES),
                )

                print(results)
                print("All Stellar nodes should be starting in the background.")

                # Give nodes time to authenticate and start the client listener.
                time.sleep(40)

                # Start only required client VMs for this load point.
                active_client_indices = [
                    NUM_NODES + j for j in range(active_clients)
                ]

                results = run_parallel(
                    lambda i: run_stellar_client(
                        i,
                        client_max_in_flight,
                        client_threads,
                    ),
                    active_client_indices,
                    max_workers=active_clients,
                )

                print(results)
                print(
                    f"Started {active_clients} client VM(s), "
                    f"each with {client_threads} client threads. "
                    f"Total client threads = {total_client_threads}. "
                    f"Aggregate max in-flight = {aggregate_max_in_flight}."
                )

                # Let the system run.
                time.sleep(CLIENT_WAIT_AFTER_START_SEC)

                # Stop all nodes and clients.
                results = run_parallel(
                    kill_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # --------------------------------------------------------
                # Save logs.
                # --------------------------------------------------------
                remote_base_folder = "/home/tejas/stellar-private"

                local_base_destination = (
                    "/home/tejas/work/experiments/shabdiz/"
                    + f"CollectionVSNetworkLatency_{max_collect_attempts}_"
                    + f"remote_{remote_replica_zone}_"
                    + f"nodes_{NUM_NODES}_{run_label}"
                )

                Path(local_base_destination).mkdir(parents=True, exist_ok=True)

                with open(Path(local_base_destination) / "run_config.txt", "w") as f:
                    f.write(f"experiment=CollectionVSNetworkLatency\n")
                    f.write(f"num_nodes={NUM_NODES}\n")
                    f.write(f"max_collect_attempts={max_collect_attempts}\n")
                    f.write(f"force_collect_after_sec=100\n")
                    f.write(f"base_replica_zone={BASE_REPLICA_ZONE}\n")
                    f.write(f"remote_replica_zone={remote_replica_zone}\n")
                    f.write(f"client_zone={CLIENT_ZONE}\n")
                    f.write(f"replicas_0_to_7_zone={BASE_REPLICA_ZONE}\n")
                    f.write(f"replicas_8_to_15_zone={remote_replica_zone}\n")
                    f.write(f"client_instance=tsm-sc-{NUM_NODES:03}\n")
                    f.write(f"active_clients={active_clients}\n")
                    f.write(f"client_threads_per_vm={client_threads}\n")
                    f.write(f"total_client_threads={total_client_threads}\n")
                    f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
                    f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
                    f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
                    f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
                    f.write(f"leader_ip={node1_ip}\n")
                    f.write(f"machine_type={machine_type}\n")
                    f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

                def copy_folder_from_instance(i):
                    inst_zone = get_zone_for_instance(i, remote_replica_zone)

                    instance_name = f"tsm-sc-{i:03}"

                    node_number = i + 1
                    node_folder = f"node{node_number}"

                    remote_source_path = posixpath.join(remote_base_folder, node_folder)

                    local_destination_path = Path(local_base_destination) / instance_name
                    local_destination_path.mkdir(parents=True, exist_ok=True)

                    remote_source = f"{instance_name}:{remote_source_path}"

                    command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

                    print(
                        f"Executing command to copy {node_folder} "
                        f"from {instance_name}: {command}"
                    )

                    output = subprocess.call(command, shell=True)

                    print(f"Copy from {instance_name} finished with exit code: {output}")

                    return (instance_name, output)

                node_copy_results = run_parallel(
                    copy_folder_from_instance,
                    COPY_NODE_INDICES,
                    max_workers=min(3, len(COPY_NODE_INDICES)),
                )

                def copy_client_log(i):
                    inst_zone = get_zone_for_instance(i, remote_replica_zone)

                    instance_name = f"tsm-sc-{i:03}"
                    client_id = i - NUM_NODES

                    remote_source = (
                        f"{instance_name}:/home/tejas/stellar-private/"
                        f"stellar-client-{client_id}.log"
                    )

                    local_destination_path = Path(local_base_destination)
                    local_destination_path.mkdir(parents=True, exist_ok=True)

                    command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

                    print(f"Copying client log from {instance_name}...")

                    output = subprocess.call(command, shell=True)

                    print(f"Copy finished with exit code: {output}")

                    return (instance_name, output)

                client_copy_results = run_parallel(
                    copy_client_log,
                    active_client_indices,
                    max_workers=active_clients,
                )

                print("\n--- Summary of Download Results ---")
                print("Node log copies:", node_copy_results)
                print("Client log copies:", client_copy_results)
                print(f"Saved run to: {local_base_destination}")

        finally:
            if DELETE_AFTER_EACH_SUBRUN:
                print(
                    "\n🧹 Deleting VMs after completed/attempted subrun: "
                    f"remote_replica_zone={remote_replica_zone}, "
                    f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
                )
                delete_all_tsm_instances(project)

In [ ]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

In [ ]:
#             # Clean remote stellar-private.
#             results = run_parallel(
#                 clean_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Copy fresh stellar-private config to all active machines.
#             results = run_parallel(
#                 copy_folder_to_instance,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Kill old processes on all active machines.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Start consensus replicas.
#             results = run_parallel(
#                 run_stellar_private,
#                 range(num_nodes),
#                 max_workers=min(48, num_nodes),
#             )

#             print(results)
#             print("All Stellar nodes should be starting in the background.")

#             # Give nodes time to authenticate and start the client listener.
#             time.sleep(60)

#             # Start only required client VMs for this load point.
#             active_client_indices = [
#                 num_nodes + j for j in range(active_clients)
#             ]

#             results = run_parallel(
#                 lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print(results)
#             print(
#                 f"Started {active_clients} client VM(s), "
#                 f"each with {client_threads} client threads. "
#                 f"Total client threads = {total_client_threads}. "
#                 f"Aggregate max in-flight = {aggregate_max_in_flight}."
#             )

#             # Let the system run before forcing failures.
#             time.sleep(CLIENT_WAIT_AFTER_START_SEC)


#             # Stop all nodes and clients.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # -----------------------------------------------------------------
#             # Save logs.
#             # -----------------------------------------------------------------
#             remote_base_folder = "/home/tejas/stellar-private"

#             local_base_destination = (
#                 "/home/tejas/work/experiments/shabdiz/"
#                 + f"Collection_{max_collect_attempts}_"
#                 + f"nodes_{num_nodes}_{run_label}"
#             )

#             Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#             # Save run metadata.
#             with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#                 f.write(f"num_nodes={num_nodes}\n")
#                 f.write(f"max_collect_attempts={max_collect_attempts}\n")
#                 f.write(f"force_collect_after_sec=100\n")
#                 f.write(f"active_clients={active_clients}\n")
#                 f.write(f"client_threads_per_vm={client_threads}\n")
#                 f.write(f"total_client_threads={total_client_threads}\n")
#                 f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#                 f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#                 f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#                 f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#                 f.write(f"leader_ip={node1_ip}\n")
#                 f.write(f"machine_type={machine_type}\n")
#                 f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#             def copy_folder_from_instance(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"

#                 node_number = i + 1
#                 node_folder = f"node{node_number}"

#                 remote_source_path = posixpath.join(remote_base_folder, node_folder)

#                 local_destination_path = Path(local_base_destination) / instance_name
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 remote_source = f"{instance_name}:{remote_source_path}"

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#                 print(
#                     f"Executing command to copy {node_folder} "
#                     f"from {instance_name}: {command}"
#                 )

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy from {instance_name} finished with exit code: {output}")

#                 return (instance_name, output)

#             # Copy all replica logs.
#             node_copy_results = run_parallel(
#                 copy_folder_from_instance,
#                 range(3),
#                 max_workers=min(48, max(1, num_nodes)),
#             )

#             def copy_client_log(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"
#                 client_id = i - num_nodes

#                 remote_source = (
#                     f"{instance_name}:/home/tejas/stellar-private/"
#                     f"stellar-client-{client_id}.log"
#                 )

#                 local_destination_path = Path(local_base_destination)
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#                 print(f"Copying client log from {instance_name}...")

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy finished with exit code: {output}")

#                 return (instance_name, output)

#             client_copy_results = run_parallel(
#                 copy_client_log,
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print("\n--- Summary of Download Results ---")
#             print("Node log copies:", node_copy_results)
#             print("Client log copies:", client_copy_results)
#             print(f"Saved run to: {local_base_destination}")

In [ ]:

# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# import re
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Fixed read-ratio experiment configuration
# # =============================================================================

# # Fixed system size for performance-vs-read-ratio experiment.
# NUM_NODES = 8

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = 8, client is tsm-sc-008.
# n_clients = 1

# # Read-ratio workloads. These must already exist in CustomYCSBWorkload.h:
# # WORKLOAD_R0, WORKLOAD_R25, WORKLOAD_R50, WORKLOAD_R75, WORKLOAD_R100
# READ_RATIO_WORKLOADS = [
#     # {"read_ratio": 0,   "workload": "WORKLOAD_R0"},
#     {"read_ratio": 10,   "workload": "WORKLOAD_R10"},
#     # {"read_ratio": 25,  "workload": "WORKLOAD_R25"},
#     # {"read_ratio": 50,  "workload": "WORKLOAD_R50"},
#     # {"read_ratio": 75,  "workload": "WORKLOAD_R75"},
#     # {"read_ratio": 20,  "workload": "WORKLOAD_R20"},
#     {"read_ratio": 40,  "workload": "WORKLOAD_R40"},
#     # {"read_ratio": 60,  "workload": "WORKLOAD_R60"},
#     # {"read_ratio": 80,  "workload": "WORKLOAD_R80"},
#     # {"read_ratio": 100, "workload": "WORKLOAD_R100"},
# ]

# # The file containing:
# #   static YCSBWorkload currentWorkload = WORKLOAD_R50;
# YCSB_HEADER_LOCAL = Path("src/overlay/CustomYCSBWorkload.h")

# # Start clean before the first read-ratio run.
# DELETE_BEFORE_FIRST_RUN = True

# # Delete all VMs after the final read-ratio run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile stellar-core only in the first read-ratio subrun.
# # The client is compiled every subrun because CustomYCSBWorkload.h changes.
# COMPILE_STELLAR_CORE_ONLY_FIRST_RUN = True

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 210
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Same fixed client load as your previous experiment.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]

# # GCP settings.
# project = "research-488322"
# zone = "us-central1-c"
# machine_type = "e2-standard-2"
# image_family = "tsm-sc-family"  # your custom image
# subnet = "default"
# gcp_username = "tejas"

# # latencies: 50, 90 150, 210
# default_region = ["us-central1-a"]
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
# regions = ["us-central1-a", "us-central1-a", "us-central1-a", "us-central1-a"]

# zone_no = 0


# def get_zone_for_instance(i):
#     # Keep the same placement rule as your previous script.
#     if i < int(NUM_NODES / 2):
#         return default_region[0]
#     else:
#         return regions[zone_no]


# def set_local_ycsb_workload(workload_name):
#     """
#     Updates this line in src/overlay/CustomYCSBWorkload.h:

#         static YCSBWorkload currentWorkload = WORKLOAD_R50;

#     to the requested workload.
#     """
#     if not YCSB_HEADER_LOCAL.exists():
#         raise FileNotFoundError(
#             f"Could not find {YCSB_HEADER_LOCAL}. "
#             "Run this script from the stellar-core repo root."
#         )

#     text = YCSB_HEADER_LOCAL.read_text()

#     new_text, count = re.subn(
#         r"static\s+YCSBWorkload\s+currentWorkload\s*=\s*WORKLOAD_[A-Za-z0-9_]+\s*;",
#         f"static YCSBWorkload currentWorkload = {workload_name};",
#         text,
#         count=1,
#     )

#     if count != 1:
#         raise RuntimeError(
#             "Could not find exactly one currentWorkload line in "
#             f"{YCSB_HEADER_LOCAL}"
#         )

#     YCSB_HEADER_LOCAL.write_text(new_text)
#     print(f"✅ Updated {YCSB_HEADER_LOCAL}: currentWorkload = {workload_name}")


# def fetch_existing_instances():
#     fetch_cmd = f"""
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={project} \
#         --quiet
#     """

#     print(f"🗑️ Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def parse_tsm_index(name):
#     return int(name.rsplit("-", 1)[1])


# def run_command(command):
#     print(f"Running: {command}")
#     return subprocess.call(command, shell=True)


# def kill_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas/stellar-private 2>/dev/null || true; \
# sudo pkill -9 stellar-core || true; sudo pkill -9 shab_client || true; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def clean_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def copy_folder_to_instance(
#     i,
#     source_folder="/home/tejas/stellar-private",
#     destination_path="/home/tejas/stellar-private",
# ):
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}" """

#     print(f"Executing command for {instance_name}: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Command for {instance_name} finished with exit code: {output}")

#     return (instance_name, output)


# def run_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     node_number = i + 1
#     instance_name = f"tsm-sc-{i:03}"

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Return code for {instance_name}: {output}")
#     return output


# def git_pull_stellar(i):
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def compile_stellar_and_client(i):
#     """
#     First subrun only.
#     This compiles both shab_client and stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def copy_ycsb_header_to_instance(i):
#     """
#     Later subruns only.
#     Copy changed CustomYCSBWorkload.h to the VM source tree.
#     This keeps the source file in sync. We only recompile the client binary after this.
#     """
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     remote_path = (
#         f"{instance_name}:/home/tejas/stellar-core/src/overlay/"
#         f"{YCSB_HEADER_LOCAL.name}"
#     )

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{YCSB_HEADER_LOCAL}" "{remote_path}" """

#     print(f"Copying workload header to {instance_name}: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Header copy to {instance_name} finished with exit code: {output}")
#     return (instance_name, output)


# def compile_client_only(i):
#     """
#     Every subrun after the first.
#     This recompiles only shab_client, not stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd /home/tejas/stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# # =============================================================================
# # VM setup
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()

#     print("\n➡ Existing instances to delete before first run:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     if instances:
#         run_parallel(
#             delete_instance,
#             instances,
#             max_workers=min(32, len(instances)),
#         )
#         print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#     else:
#         print("\n✔ No existing tsm-sc-* instances found before first run.\n")


# # Create required replica/client machines.
# existing_instances = fetch_existing_instances()
# existing_names = {name for name, _ in existing_instances}

# commands = []

# # Replicas: tsm-sc-000 ... tsm-sc-007
# for i in range(NUM_NODES):
#     instance_name = f"tsm-sc-{i:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing replica {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# # Client: tsm-sc-008
# for i in range(n_clients):
#     client_idx = NUM_NODES + i
#     instance_name = f"tsm-sc-{client_idx:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing client {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(client_idx)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# if commands:
#     run_parallel(
#         run_command,
#         commands,
#         max_workers=min(48, len(commands)),
#     )

#     print("All missing instances launched.")

#     # Give GCP/SSH a little time after VM creation.
#     time.sleep(30)
# else:
#     print("✔ All required instances already exist; no VM creation needed.")


# # Get sorted node and client IPs.
# ip_cmd = f"""
# gcloud compute instances list \
#     --project={project} \
#     --filter="name~'^tsm-sc-'" \
#     --sort-by=name \
#     --format="value(name,zone,networkInterfaces[0].networkIP)"
# """

# ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

# instance_records = []

# for line in ip_output.splitlines():
#     if line.strip():
#         name, inst_zone, ip = line.split()
#         idx = int(name.rsplit("-", 1)[1])
#         instance_records.append((idx, name, inst_zone, ip))

# instance_records.sort()

# node_records = [r for r in instance_records if r[0] < NUM_NODES]
# client_records = [r for r in instance_records if NUM_NODES <= r[0] < NUM_NODES + n_clients]

# if len(node_records) != NUM_NODES:
#     raise RuntimeError(
#         f"Expected {NUM_NODES} replica nodes, but found {len(node_records)}: {node_records}"
#     )

# if len(client_records) != n_clients:
#     raise RuntimeError(
#         f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#     )

# iplist = [r[3] for r in node_records]

# with open("tsm_ips.txt", "w") as f:
#     for ip in iplist:
#         f.write(ip + "\n")

# print("🎯 Node IPs:", iplist)
# print("🎯 Client instances:", client_records)

# node1_ip = iplist[0]
# print(f"Clients will connect to leader/node1 at: {node1_ip}")

# all_instance_indices = list(range(NUM_NODES + n_clients))


# # =============================================================================
# # Read-ratio experiment loop
# # =============================================================================

# for workload_idx, workload_cfg in enumerate(READ_RATIO_WORKLOADS):
#     read_ratio = workload_cfg["read_ratio"]
#     workload_name = workload_cfg["workload"]

#     print("\n" + "#" * 100)
#     print(
#         f"Starting read-ratio experiment: "
#         f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, workload={workload_name}"
#     )
#     print("#" * 100)

#     # -------------------------------------------------------------------------
#     # Change CustomYCSBWorkload.h for this subrun.
#     # -------------------------------------------------------------------------
#     set_local_ycsb_workload(workload_name)

#     # -------------------------------------------------------------------------
#     # First subrun:
#     #   - commit/push changed workload
#     #   - git pull on every VM
#     #   - compile shab_client + stellar-core on every VM
#     #
#     # Later subruns:
#     #   - copy changed header to VMs
#     #   - compile only shab_client on client VM(s)
#     # -------------------------------------------------------------------------
#     first_subrun = (workload_idx == 0)

#     if first_subrun:
#         subprocess.call(
#             f'git add .; git commit -m "read ratio workload {workload_name}" || true; git push',
#             shell=True,
#         )

#         # Optional local build, preserved from your earlier script.
#         subprocess.call("make -j8", shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             git_pull_stellar,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#         results = run_parallel(
#             compile_stellar_and_client,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)
#     else:
#         # No git pull and no stellar-core compilation after the first subrun.
#         # The server binary is unchanged. The client workload changes through
#         # CustomYCSBWorkload.h, so update source and recompile shab_client.
#         results = run_parallel(
#             copy_ycsb_header_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path("../stellar-private")
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     subprocess.call(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True,
#     )

#     # -------------------------------------------------------------------------
#     # Fixed client load loop.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = (
#             active_clients * client_threads * client_max_in_flight
#         )

#         active_client_indices = [
#             NUM_NODES + j for j in range(active_clients)
#         ]

#         # Compile the client every subrun. On the first subrun it was already
#         # compiled by compile_stellar_and_client(), but compiling it again is
#         # cheap and guarantees the workload header is reflected in shab_client.
#         results = run_parallel(
#             compile_client_only,
#             active_client_indices,
#             max_workers=active_clients,
#         )
#         print(results)

#         run_label = (
#             f"read_ratio_{read_ratio}_"
#             f"{workload_name}_"
#             f"nodes_{NUM_NODES}_"
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(
#             f"🚀 Starting read-ratio run: "
#             f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, "
#             f"workload={workload_name}, {run_label}"
#         )
#         print("=" * 80)

#         results = run_parallel(
#             clean_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             copy_folder_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(NUM_NODES),
#             max_workers=min(48, NUM_NODES),
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial write batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SHABDIZ_vs_read_ratio_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={NUM_NODES}\n")
#             f.write(f"read_ratio_percent={read_ratio}\n")
#             f.write(f"workload={workload_name}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"client_wait_after_start_sec={CLIENT_WAIT_AFTER_START_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")
#             f.write(f"compile_stellar_core_this_subrun={first_subrun}\n")
#             f.write("client_compiled_this_subrun=true\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}" """

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, NUM_NODES)) to range(NUM_NODES) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, NUM_NODES)),
#             max_workers=min(48, max(1, min(3, NUM_NODES))),
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log" """

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")


# # =============================================================================
# # Cleanup after all read-ratio subruns
# # =============================================================================

# # if DELETE_ALL_AFTER_FINAL_RUN:
# #     instances = fetch_existing_instances()

# #     print("\n➡ All read-ratio runs complete. Deleting all remaining tsm-sc-* VMs:")
# #     for name, inst_zone in instances:
# #         print(f"  - {name} ({inst_zone})")

# #     if instances:
# #         run_parallel(
# #             delete_instance,
# #             instances,
# #             max_workers=min(32, len(instances)),
# #         )
# #         print(f"\n🧹 Deleted {len(instances)} instance(s).\n")
# #     else:
# #         print("\n✔ No instances to delete.\n")
# # else:
# #     print("\n✔ All read-ratio runs complete. Leaving VMs running.")


#  Memory Exp

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

# #     if do_compile_this_run:
# #         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

# #         n_collection = 100
# #         subprocess.call('make -j8', shell=True)

# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def git_pull_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # git pull"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             git_pull_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )
# #         print(results)

# #         # ---------------------------------------------------------------------
# #         # Compile only on the first/largest run.
# #         # Since later runs reuse a subset of these VMs, no recompilation is needed.
# #         # ---------------------------------------------------------------------
# #         def compile_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # g++ -O2 -std=c++17 -pthread \
# # -I/home/tejas/stellar-core/src \
# # /home/tejas/stellar-core/shab_client.cpp \
# # -o /home/tejas/stellar-core/shab_client; \
# # make -j16; \
# # cd; \
# # sudo rm -rf stellar-private"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             compile_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )


        
# #         print(results)
# #     else:
# #         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

# #     # -------------------------------------------------------------------------
# #     # Generate stellar-private configs locally using only replica IPs.
# #     # -------------------------------------------------------------------------
# #     stellar_private_path = Path('../stellar-private')
# #     if stellar_private_path.exists():
# #         shutil.rmtree(stellar_private_path)
# #     stellar_private_path.mkdir()

# #     subprocess.call(
# #         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     subprocess.call(
# #         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
# #         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     # Enable custom message only on leader.
# #     line_to_add = "SEND_CUSTOM_MESSAGE=true"
# #     target_file = "../stellar-private/node1/stellar-core.cfg"

# #     subprocess.call(
# #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #         shell=True
# #     )

# #     # Optional memory profiling on node2.
# #     if num_nodes >= 2:
# #         target_file = "../stellar-private/node2/stellar-core.cfg"
# #         line_to_add = "MEMORY_PROF=true"

# #         subprocess.call(
# #             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #             shell=True
# #         )

# #     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

# #     # -------------------------------------------------------------------------
# #     # Throughput/latency experiment loop.
# #     # Fixed offered load for scalability:
# #     # active_clients=1, client_threads=2, max_in_flight=100.
# #     # Aggregate max in-flight = 200.
# #     # -------------------------------------------------------------------------
# #     for load in LOAD_POINTS:
# #         active_clients = load["active_clients"]
# #         client_threads = load["client_threads"]
# #         client_max_in_flight = load["max_in_flight"]

# #         if active_clients > n_clients:
# #             raise RuntimeError(
# #                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
# #             )

# #         total_client_threads = active_clients * client_threads
# #         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

# #         run_label = (
# #             f"clients_{active_clients}_threads_{client_threads}_"
# #             f"inflight_{client_max_in_flight}_"
# #             f"total_threads_{total_client_threads}_"
# #             f"total_inflight_{aggregate_max_in_flight}"
# #         )

# #         print("\n" + "=" * 80)
# #         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
# #         print("=" * 80)

# #         def clean_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             remote_command = f"""\
# # cd /home/tejas; \
# # sudo rm -rf stellar-private; \
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")
# #             output = subprocess.call(command, shell=True)
# #             print(f"Return code for tsm-sc-{i:03}: {output}")
# #             return output

# #         results = run_parallel(
# #             clean_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def copy_folder_to_instance(
# #             i,
# #             source_folder="/home/tejas/stellar-private",
# #             destination_path="/home/tejas/stellar-private"
# #         ):
# #             inst_zone = get_zone_for_instance(i)
# #             instance_name = f"tsm-sc-{i:03}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

# #             print(f"Executing command for {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Command for {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         results = run_parallel(
# #             copy_folder_to_instance,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def run_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             node_number = i + 1
# #             instance_name = f"tsm-sc-{i:03}"

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# # > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Kill old processes on all replica and client machines.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         # Start consensus replicas.
# #         results = run_parallel(
# #             run_stellar_private,
# #             range(num_nodes),
# #             max_workers=min(48, num_nodes)
# #         )

# #         print(results)
# #         print("All Stellar nodes should be starting in the background.")

# #         # Give nodes time to authenticate and start the client listener.
# #         time.sleep(80)

# #         def run_stellar_client(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# # {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# # {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# # > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing client command: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Start only the required number of client VMs for this load point.
# #         active_client_indices = [
# #             num_nodes + j for j in range(active_clients)
# #         ]

# #         results = run_parallel(
# #             run_stellar_client,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print(results)
# #         print(
# #             f"Started {active_clients} client VM(s), "
# #             f"each with {client_threads} client threads. "
# #             f"Total client threads = {total_client_threads}. "
# #             f"Aggregate max in-flight = {aggregate_max_in_flight}."
# #         )

# #         # Wait for the duration run to produce stable per-second client logs.
# #         # The clients may wait forever on final partial batches, so we kill them after this.
# #         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

# #         # Stop all nodes and clients.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         remote_base_folder = "/home/tejas/stellar-private"

# #         local_base_destination = (
# #             "/home/tejas/work/experiments/shabdiz/"
# #             + f"memory_no_cleanup_{num_nodes}_{run_label}"
# #         )

# #         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

# #         # Save run metadata.
# #         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
# #             f.write(f"num_nodes={num_nodes}\n")
# #             f.write(f"active_clients={active_clients}\n")
# #             f.write(f"client_threads_per_vm={client_threads}\n")
# #             f.write(f"total_client_threads={total_client_threads}\n")
# #             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
# #             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
# #             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
# #             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
# #             f.write(f"leader_ip={node1_ip}\n")
# #             f.write(f"machine_type={machine_type}\n")
# #             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

# #         def copy_folder_from_instance(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"

# #             node_number = i + 1
# #             node_folder = f"node{node_number}"

# #             remote_source_path = posixpath.join(remote_base_folder, node_folder)

# #             local_destination_path = Path(local_base_destination) / instance_name
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             remote_source = f"{instance_name}:{remote_source_path}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{remote_source}" "{local_destination_path}"'''

# #             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy from {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         # Copy only a few node logs to reduce time.
# #         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
# #         node_copy_results = run_parallel(
# #             copy_folder_from_instance,
# #             range((num_nodes)),
# #             max_workers=min(48, max(1, min(3, num_nodes)))
# #         )

# #         def copy_client_log(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_source = (
# #                 f"{instance_name}:/home/tejas/stellar-private/"
# #                 f"stellar-client-{client_id}.log"
# #             )

# #             local_destination_path = Path(local_base_destination)
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

# #             print(f"Copying client log from {instance_name}...")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy finished with exit code: {output}")

# #             return (instance_name, output)

# #         client_copy_results = run_parallel(
# #             copy_client_log,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print("\n--- Summary of Download Results ---")
# #         print("Node log copies:", node_copy_results)
# #         print("Client log copies:", client_copy_results)
# #         print(f"Saved run to: {local_base_destination}")



#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     if num_nodes >= 2:
#         target_file = "../stellar-private/node2/stellar-core.cfg"
#         line_to_add = "MEMORY_PROF=true"

#         subprocess.call(
#             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#             shell=True
#         )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"memory_withv3_cleanup_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# PBFT Test

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output




#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )


#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output



#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(40)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SCP_test_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# SCP

In [ ]:
# import subprocess
# import concurrent.futures
# import shutil
# import time
# import re
# from pathlib import Path
# from datetime import datetime


# # =============================================================================
# # Basic helpers
# # =============================================================================

# def run_shell(command, check=False):
#     print(f"\n$ {command}")
#     rc = subprocess.call(command, shell=True)
#     if check and rc != 0:
#         raise RuntimeError(f"Command failed with code {rc}: {command}")
#     return rc


# def run_shell_output(command):
#     print(f"\n$ {command}")
#     return subprocess.check_output(command, shell=True).decode()


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Experiment config
# # =============================================================================

# PROJECT = "research-488322"
# ZONE = "us-central1-b"
# MACHINE_TYPE = "e2-standard-2"
# IMAGE_FAMILY = "tsm-sc-family"
# SUBNET = "default"
# GCP_USERNAME = "tejas"

# # Run in decreasing order so we can delete unused nodes after each subrun.
# NUM_NODES_LIST = [48, 32, 16, 8, 4]
# MAX_NUM_NODES = max(NUM_NODES_LIST)

# N_CLIENTS = 0

# # You said NUM_TEST_ACCOUNTS is already set to 100 in C++.
# NUM_TEST_ACCOUNTS = 100

# # This is only used for summary labels. It is not patched by this script.
# SCP_OPS_PER_TX = 100

# RUN_DURATION_SEC = 210
# MEASURE_START_SEC = 120
# MEASURE_END_SEC = 180

# LOCAL_PRIVATE_DIR = Path("../stellar-private")
# RESULTS_DIR = Path("/home/tejas/work/experiments/stellar_native")
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# DELETE_BEFORE_FIRST_RUN = False

# # Final cleanup deletes whatever remains after the 4-node run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile only during the first subrun.
# COMPILE_ONLY_FIRST_SUBRUN = True


# # =============================================================================
# # GCP helpers
# # =============================================================================

# def instance_name(i):
#     return f"tsm-sc-{i:03}"


# def get_zone_for_instance(i):
#     return ZONE


# def fetch_existing_instances():
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance
#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def delete_instance_by_index(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting unused node {name}")
#     return subprocess.call(cmd, shell=True)


# def delete_nodes_not_needed_next(current_num_nodes, next_num_nodes):
#     """
#     Because NUM_NODES_LIST is descending:
#       after 48-node run, delete 32..47
#       after 32-node run, delete 16..31
#       after 16-node run, delete 8..15
#       after 8-node run, delete 4..7
#     """
#     if next_num_nodes is None:
#         return

#     to_delete = list(range(next_num_nodes, current_num_nodes))

#     if not to_delete:
#         return

#     print(
#         f"Deleting nodes no longer needed for future subruns: "
#         f"{instance_name(to_delete[0])} to {instance_name(to_delete[-1])}"
#     )

#     run_parallel(
#         delete_instance_by_index,
#         to_delete,
#         max_workers=min(32, len(to_delete)),
#     )


# def create_instance(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {name} \
#         --project={PROJECT} \
#         --zone={inst_zone} \
#         --machine-type={MACHINE_TYPE} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={SUBNET} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={IMAGE_FAMILY},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     print(f"Creating {name}")
#     return subprocess.call(cmd, shell=True)


# def ensure_instances():
#     existing = fetch_existing_instances()
#     existing_names = {name for name, _ in existing}

#     needed_indices = list(range(MAX_NUM_NODES + N_CLIENTS))
#     missing = [i for i in needed_indices if instance_name(i) not in existing_names]

#     if not missing:
#         print("All required instances already exist.")
#         return

#     print("Missing instances:", [instance_name(i) for i in missing])
#     run_parallel(create_instance, missing, max_workers=min(32, len(missing)))

#     print("Waiting after VM creation...")
#     time.sleep(30)


# def get_node_records(num_nodes):
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()

#     records = []
#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             records.append((idx, name, inst_zone, ip))

#     records.sort()
#     node_records = [r for r in records if r[0] < num_nodes]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} nodes, found {len(node_records)}: {node_records}"
#         )

#     return node_records


# def kill_stellar(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo pkill -9 stellar-core || true; \
# sudo pkill -9 shab_client || true; \
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def git_pull_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# git pull
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def compile_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# make -j2; \
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def clean_remote_private(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def copy_private_to_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# --recurse "/home/tejas/stellar-private" "{name}:/home/tejas/stellar-private"
# """
#     return subprocess.call(cmd, shell=True)


# def run_stellar_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def collect_node_log(i, local_run_dir):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     local_run_dir.mkdir(parents=True, exist_ok=True)
#     local_path = local_run_dir / f"{name}-node{node_number}-stellar-core.log"

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# "{name}:/home/tejas/stellar-private/node{node_number}/stellar-core.log" \
# "{local_path}"
# """
#     return subprocess.call(cmd, shell=True)


# # =============================================================================
# # Config generation
# # =============================================================================

# def write_tsm_ips(node_ips):
#     with open("tsm_ips.txt", "w") as f:
#         for ip in node_ips:
#             f.write(ip + "\n")


# def generate_stellar_private_configs(num_nodes):
#     if LOCAL_PRIVATE_DIR.exists():
#         shutil.rmtree(LOCAL_PRIVATE_DIR)

#     LOCAL_PRIVATE_DIR.mkdir(parents=True)

#     run_shell(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     run_shell(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     # Only node1 injects native Stellar workload.
#     # ENABLE_SCP_TRACKING is already true in C++.
#     for node_number in range(1, num_nodes + 1):
#         cfg = LOCAL_PRIVATE_DIR / f"node{node_number}" / "stellar-core.cfg"

#         if not cfg.exists():
#             raise FileNotFoundError(f"Missing config: {cfg}")

#         lines = cfg.read_text().splitlines()
#         lines = [
#             line for line in lines
#             if not line.strip().startswith("SEND_CUSTOM_MESSAGE=")
#         ]

#         if node_number == 1:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=true")
#         else:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=false")

#         cfg.write_text("\n".join(lines) + "\n")


# # =============================================================================
# # Log parsing
# # =============================================================================

# def parse_ts(line):
#     # Example:
#     # 2026-06-30T00:09:11.842 GDQOM [Ledger INFO] Got consensus...
#     m = re.match(r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?)", line)
#     if not m:
#         return None

#     try:
#         return datetime.fromisoformat(m.group(1))
#     except ValueError:
#         return None


# def parse_committed_ops(log_path, start_sec=MEASURE_START_SEC, end_sec=MEASURE_END_SEC):
#     entries = []

#     with open(log_path, "r", errors="ignore") as f:
#         for line in f:
#             if "Got consensus:" not in line:
#                 continue

#             ts = parse_ts(line)
#             if ts is None:
#                 continue

#             m_ops = re.search(r"ops=(\d+)", line)
#             m_txs = re.search(r"txs=(\d+)", line)

#             if not m_ops:
#                 continue

#             ops = int(m_ops.group(1))
#             txs = int(m_txs.group(1)) if m_txs else 0

#             entries.append((ts, txs, ops, line.rstrip()))

#     if not entries:
#         return {
#             "ops": 0,
#             "txs": 0,
#             "throughput_ops_s": 0.0,
#             "avg_ops_per_ledger": 0.0,
#             "avg_txs_per_ledger": 0.0,
#             "min_ops_per_ledger": 0,
#             "max_ops_per_ledger": 0,
#             "min_txs_per_ledger": 0,
#             "max_txs_per_ledger": 0,
#             "num_ledgers": 0,
#             "first_ts": None,
#             "last_ts": None,
#         }

#     t0 = entries[0][0]
#     selected = []

#     for ts, txs, ops, line in entries:
#         elapsed = (ts - t0).total_seconds()
#         if start_sec <= elapsed <= end_sec:
#             selected.append((ts, txs, ops, line))

#     total_ops = sum(ops for _, _, ops, _ in selected)
#     total_txs = sum(txs for _, txs, _, _ in selected)
#     num_ledgers = len(selected)
#     window = max(1.0, end_sec - start_sec)

#     ops_values = [ops for _, _, ops, _ in selected]
#     txs_values = [txs for _, txs, _, _ in selected]

#     return {
#         "ops": total_ops,
#         "txs": total_txs,
#         "throughput_ops_s": total_ops / window,
#         "avg_ops_per_ledger": total_ops / num_ledgers if num_ledgers else 0.0,
#         "avg_txs_per_ledger": total_txs / num_ledgers if num_ledgers else 0.0,
#         "min_ops_per_ledger": min(ops_values) if ops_values else 0,
#         "max_ops_per_ledger": max(ops_values) if ops_values else 0,
#         "min_txs_per_ledger": min(txs_values) if txs_values else 0,
#         "max_txs_per_ledger": max(txs_values) if txs_values else 0,
#         "num_ledgers": num_ledgers,
#         "first_ts": selected[0][0].isoformat() if selected else None,
#         "last_ts": selected[-1][0].isoformat() if selected else None,
#     }


# def write_summary(run_dir, num_nodes, leader_log):
#     summary = parse_committed_ops(leader_log)

#     summary_path = run_dir / "summary.txt"

#     with open(summary_path, "w") as f:
#         f.write(f"num_nodes={num_nodes}\n")
#         f.write(f"num_accounts={NUM_TEST_ACCOUNTS}\n")
#         f.write(f"ops_per_tx={SCP_OPS_PER_TX}\n")
#         f.write(f"measurement_window={MEASURE_START_SEC}-{MEASURE_END_SEC}s\n")
#         f.write(f"committed_ops={summary['ops']}\n")
#         f.write(f"committed_txs={summary['txs']}\n")
#         f.write(f"throughput_ops_s={summary['throughput_ops_s']:.2f}\n")
#         f.write(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}\n")
#         f.write(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}\n")
#         f.write(f"min_ops_per_ledger={summary['min_ops_per_ledger']}\n")
#         f.write(f"max_ops_per_ledger={summary['max_ops_per_ledger']}\n")
#         f.write(f"min_txs_per_ledger={summary['min_txs_per_ledger']}\n")
#         f.write(f"max_txs_per_ledger={summary['max_txs_per_ledger']}\n")
#         f.write(f"num_ledgers={summary['num_ledgers']}\n")
#         f.write(f"first_ts={summary['first_ts']}\n")
#         f.write(f"last_ts={summary['last_ts']}\n")

#     print("\n" + "=" * 80)
#     print(f"RESULT num_nodes={num_nodes}")
#     print(f"committed_ops={summary['ops']}")
#     print(f"committed_txs={summary['txs']}")
#     print(f"throughput_ops_s={summary['throughput_ops_s']:.2f}")
#     print(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}")
#     print(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}")
#     print(f"min_ops_per_ledger={summary['min_ops_per_ledger']}")
#     print(f"max_ops_per_ledger={summary['max_ops_per_ledger']}")
#     print(f"num_ledgers={summary['num_ledgers']}")
#     print(f"summary={summary_path}")
#     print("=" * 80 + "\n")

#     return summary


# import os
# os.system('git add .; git commit -m "update"; git push')

# # =============================================================================
# # Main experiment
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()
#     if instances:
#         run_parallel(delete_instance, instances, max_workers=min(32, len(instances)))

# ensure_instances()

# max_node_records = get_node_records(MAX_NUM_NODES)

# print("All available node records:")
# for r in max_node_records:
#     print(r)

# all_results = []

# os.system('git add .; git commit -m "update"; git push')

# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
#     run_name = f"stellar_native_nodes_{num_nodes}_accounts_{NUM_TEST_ACCOUNTS}"
#     run_dir = RESULTS_DIR / run_name
#     run_dir.mkdir(parents=True, exist_ok=True)

#     print("\n" + "#" * 100)
#     print(
#         f"Starting Stellar native run: "
#         f"nodes={num_nodes}, accounts={NUM_TEST_ACCOUNTS}, opsPerTx={SCP_OPS_PER_TX}"
#     )
#     print("#" * 100)

#     active_indices = list(range(num_nodes))
#     active_node_records = get_node_records(num_nodes)
#     active_node_ips = [r[3] for r in active_node_records]

#     print("Active node records:")
#     for r in active_node_records:
#         print(r)

#     print("Active node IPs:", active_node_ips)

#     # Kill all currently-existing nodes before each subrun.
#     # For first run this is 48; after deletion it becomes smaller.
#     existing_before_run = fetch_existing_instances()
#     existing_indices = []
#     for name, _ in existing_before_run:
#         if name.startswith("tsm-sc-"):
#             try:
#                 idx = int(name.rsplit("-", 1)[1])
#                 existing_indices.append(idx)
#             except ValueError:
#                 pass

#     if existing_indices:
#         print("Killing Stellar on all existing tsm-sc nodes before subrun...")
#         run_parallel(
#             kill_stellar,
#             sorted(existing_indices),
#             max_workers=min(48, len(existing_indices)),
#         )

#     # Compile only once, during the first subrun.
#     if (not COMPILE_ONLY_FIRST_SUBRUN) or (run_idx == 0):
#         print("Compiling Stellar Core on all max nodes. This happens only for first subrun.")

#         run_parallel(
#             git_pull_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )

#         run_parallel(
#             compile_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )
#     else:
#         print("Skipping compile for this subrun.")


#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Generate configs for the current node count.
#     # This rewrites tsm_ips.txt so the generated private network has exactly num_nodes validators.
#     write_tsm_ips(active_node_ips)
#     generate_stellar_private_configs(num_nodes)

#     # Clean and copy configs only to active nodes.
#     run_parallel(
#         clean_remote_private,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     run_parallel(
#         copy_private_to_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Start only active nodes.
#     run_parallel(
#         run_stellar_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print(f"Nodes started. Running for {RUN_DURATION_SEC} seconds...")
#     time.sleep(RUN_DURATION_SEC)

#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print("Collecting logs...")
#     run_parallel(
#         lambda i: collect_node_log(i, run_dir),
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     leader_log = run_dir / "tsm-sc-000-node1-stellar-core.log"

#     if leader_log.exists():
#         result = write_summary(run_dir, num_nodes, leader_log)
#         result["num_nodes"] = num_nodes
#         all_results.append(result)
#     else:
#         print(f"WARNING: leader log not found: {leader_log}")

#     # Delete nodes that are no longer needed for the next smaller subrun.
#     if run_idx + 1 < len(NUM_NODES_LIST):
#         next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#         delete_nodes_not_needed_next(num_nodes, next_num_nodes)


# # =============================================================================
# # Final CSV summary
# # =============================================================================

# csv_path = RESULTS_DIR / f"stellar_native_accounts_{NUM_TEST_ACCOUNTS}_nodes_sweep_summary.csv"

# with open(csv_path, "w") as f:
#     f.write(
#         "num_nodes,num_accounts,committed_ops,committed_txs,throughput_ops_s,"
#         "avg_ops_per_ledger,avg_txs_per_ledger,"
#         "min_ops_per_ledger,max_ops_per_ledger,"
#         "min_txs_per_ledger,max_txs_per_ledger,"
#         "num_ledgers,first_ts,last_ts\n"
#     )

#     for r in all_results:
#         f.write(
#             f"{r['num_nodes']},"
#             f"{NUM_TEST_ACCOUNTS},"
#             f"{r['ops']},"
#             f"{r['txs']},"
#             f"{r['throughput_ops_s']:.2f},"
#             f"{r['avg_ops_per_ledger']:.2f},"
#             f"{r['avg_txs_per_ledger']:.2f},"
#             f"{r['min_ops_per_ledger']},"
#             f"{r['max_ops_per_ledger']},"
#             f"{r['min_txs_per_ledger']},"
#             f"{r['max_txs_per_ledger']},"
#             f"{r['num_ledgers']},"
#             f"{r['first_ts']},"
#             f"{r['last_ts']}\n"
#         )

# print(f"\nFinal CSV summary: {csv_path}")

# if DELETE_ALL_AFTER_FINAL_RUN:
#     print("Deleting remaining tsm-sc instances after final run...")
#     instances = fetch_existing_instances()
#     tsm_instances = [
#         (name, zone)
#         for name, zone in instances
#         if name.startswith("tsm-sc-")
#     ]

#     run_parallel(
#         delete_instance,
#         tsm_instances,
#         max_workers=min(32, len(tsm_instances)),
#     )

In [ ]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")